In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Data collection and processing

In [ ]:
import sys
from pathlib import Path

# Find project root from wherever this notebook is currently running
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "PepCL_Manuscript":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find PepCL_Manuscript project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

# Allow imports like: from src... and from Data...
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
ANALYSIS_DIR = PROJECT_ROOT / "Analysis"
FIGURES_DIR = PROJECT_ROOT / "Figures"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

PROJECT_ROOT: /home/pchati/PepCL_Manuscript
DATA_DIR: /home/pchati/PepCL_Manuscript/Data


In [ ]:
import pandas as pd
import numpy as np
import os
from collections import Counter
import warnings
import seaborn as sns
import matplotlib.pyplot as plt

# supress print warnings
warnings.filterwarnings("ignore") # default, ignore, always

In [ ]:
# imports
from src.general_utils import *
from src.louvain_clustering import *
from src.log_odds import *
from src.mhcprime_inference import *

# get feature data and processed feature table
from src.get_feature_data import *

### Proteome data, MHC pseudosequences, and feature tables

#### MHC pseudosequences

 - MHC pseudosequences were collected from: https://gist.github.com/josiehong/2f9b5256ec182249272d60b225b98341.
 - These were originally derived from: https://github.com/gifford-lab/PUFFIN/blob/master/data/MHC_pseudo.dat.
 - Positions included are: 7, 9, 24, 45, 59, 62, 63, 66, 67, 69, 70, 73, 74, 76, 77, 80, 81, 84, 95, 97, 99, 114, 116, 118, 143, 147, 150, 152, 156, 158, 159, 163, 167, 171. These are derived from Figure 4 of https://pmc.ncbi.nlm.nih.gov/articles/PMC1949492/. 

In [ ]:
# get mhc pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

#### Human protein sequences

 - Human proteome was downloaded from UniProt: https://www.uniprot.org/proteomes/UP000005640. 
 - Select `Download` and options `Download only reviewed (Swiss-Prot) canonical proteins (20,416)`, `Format: TSV`, and `Compressed: Yes`.

In [ ]:
# process
canonical_human_proteins=pd.read_csv(f'{DATA_DIR}/uniprotkb_proteome_UP000005640_AND_revi_2026_05_31.tsv.gz', compression='gzip', sep="\t", index_col=0)
canonical_human_proteins.reset_index(inplace=True)
canonical_human_proteins.rename(columns={"Entry": "id", "Gene Names": "gene", "Sequence": "seq"}, inplace=True)
canonical_human_proteins=canonical_human_proteins[["id", "gene", "seq"]]
canonical_human_proteins.drop_duplicates(subset=["seq"], inplace=True)
print(canonical_human_proteins.shape)

(20338, 3)


In [ ]:
"""
Save
"""
# canonical_human_proteins.to_csv(f"{DATA_DIR}/canonical_human_proteins.csv")

'\nSave\n'

#### Swiss-Prot proteome

 - Download from: https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz.

In [ ]:
# get df and process
file_path = f"{DATA_DIR}/uniprot_sprot.fasta"
swissprot_proteome = fasta_to_dataframe(file_path)
swissprot_proteome=swissprot_proteome.drop_duplicates(subset=["seq"])
swissprot_proteome = swissprot_proteome[~swissprot_proteome['seq'].str.contains(r'[^ACDEFGHIKLMNPQRSTVWY]', regex=True)]
print(swissprot_proteome.shape)

(483737, 2)


In [ ]:
"""
Save
"""
# drop dup
# swissprot_proteome.to_csv(f"{DATA_DIR}/swissprot_proteome.csv")

'\nSave\n'

#### Processed feature table for MHCPrime

 - See the `get_feature_data.py` script in `src`.

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

In [ ]:
# save
# processed_feature_table.to_csv(f"{DATA_DIR}/processed_feature_table.csv")

### MS datasets

#### IEDB

 - Source: https://www.iedb.org/database_export_v3.php
 - Dataset: mhc_ligand_full (single_file.zip) from CSV metric exports. 
 - Location: `Data/IEDB/mhc_ligand_full.csv`.

In [ ]:
%%time
# import 
# os.listdir("Data/IEDB")

# either unzip and import like this:
# iedb=pd.read_csv("Data/IEDB/mhc_ligand_full.csv")

# or import in zip format
iedb=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/IEDB/mhc_ligand_full_single_file.zip", compression="zip")

# map columns
iedb_column_map=dict(zip(iedb.columns.tolist(), iedb.iloc[0].tolist()))

CPU times: user 58.4 s, sys: 8.01 s, total: 1min 6s
Wall time: 1min 6s


In [ ]:
"""
Full processing

 - Isolate columns of interest.
 - Rename columns.
 - Add aggregate dataset name.
 - Format alleles and labels. 
 - Filter based on canonical amino acids, MHC type, etc.
"""

# isolate and rename columns
iedb_f=iedb[["Epitope.2", "Assay.5", "MHC Restriction", "MHC Restriction.4", "Epitope.16", "Epitope.14", "Assay.1", "Epitope.4", "Epitope.5", "Reference.2"]]
iedb_f.columns=['seq', 'assay_classification','allele','hla_type','species','source_organism','method','modified_residues','modifications','pmid']
iedb_f.drop([0], inplace=True)

# isolate hla type
iedb_f=iedb_f.query("hla_type=='I'")

# filter out non-canonical seqs
iedb_f=filter_canonical_sequences(iedb_f, "seq", amino_acids='ACDEFGHIKLMNPQRSTVWY')

# get all positives (regardless of their alleles) to filter out improperly generated negatives
iedb_f_all_positive_sequences=iedb_f.query("assay_classification!='Negative'").seq.tolist()

# standardize allele format
iedb_f=standardize_allele_format(iedb_f, no_hla_prefix=False)

# create label columns
iedb_f["dataset"]=["IEDB"]*len(iedb_f)
iedb_f["label"]=iedb_f.assay_classification.replace({"Positive":1, "Positive-High":1, "Positive-Intermediate":1, "Positive-Low":1, "Negative":0})

# reset index
iedb_f.reset_index(drop=True, inplace=True)

# convert pmid to int and fill missing values w/ 0
iedb_f.pmid=iedb_f.pmid.fillna(0).astype(int)

# group methods into MS and non-MS techniques
ms_methods=["cellular MHC/mass spectrometry", "mass spectrometry", "secreted MHC/mass spectrometry"]
method_group=["MS" if a in ms_methods else "nonMS" for a in iedb_f.method]
iedb_f["method_group"]=method_group

# iedb ms pos only
iedb_f_ms_pos=iedb_f.query("method_group=='MS' and label == 1").copy()

# drop duplicates within same class
iedb_f_ms_pos.drop_duplicates(subset=["seq", "allele", "label"], inplace=True)

In [ ]:
"""
Adjust duplicate dropping order to make sure we don't accidentally remove any non-MS labeled peptides. Basically, anything eluted via MS should be labeled MS. It should be unique though across all datasets. 
"""

"\nAdjust duplicate dropping order to make sure we don't accidentally remove any non-MS labeled peptides. Basically, anything eluted via MS should be labeled MS. It should be unique though across all datasets. \n"

In [ ]:
# print stats
print_data_stats(iedb_f, allele_dist=False)
print("N alleles", iedb_f.allele.nunique())

Total: 1,401,025
Pos: 1,289,509 | 92.04%
Neg: 111,516 | 7.96%
N alleles: 211
N alleles 211


In [ ]:
# print stats
print_data_stats(iedb_f_ms_pos, allele_dist=False)
print("N alleles", iedb_f.allele.nunique())

Total: 624,794
Pos: 624,794 | 100.00%
Neg: 0 | 0.00%
N alleles: 176
N alleles 211


In [ ]:
# print stats
print_data_stats(iedb_f.query("method_group=='MS'"), allele_dist=False)
print("N alleles", iedb_f.query("method_group=='MS'").allele.nunique())

Total: 1,223,282
Pos: 1,199,784 | 98.08%
Neg: 23,498 | 1.92%
N alleles: 176
N alleles 176


In [ ]:
# print stats
print_data_stats(iedb_f.query("method_group=='nonMS'"), allele_dist=False)
print("N alleles", iedb_f.query("method_group=='nonMS'").allele.nunique())

Total: 177,743
Pos: 89,725 | 50.48%
Neg: 88,018 | 49.52%
N alleles: 135
N alleles 135


#### MHC Motif Atlas

 - Source: http://mhcmotifatlas.org/class1.
 - Datasets: All unmodified ligands.
 - Location: `Data/MHC_Motif_Atlas/data_classI_MS_Peptides_all_peptides_unmodified.txt`

In [ ]:
%%time
# import
mhcma=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/MHC_Motif_Atlas/data_classI_MS_Peptides_all_peptides_unmodified.txt", sep="\t")

CPU times: user 122 ms, sys: 297 µs, total: 122 ms
Wall time: 121 ms


In [ ]:
# format
mhcma.rename(columns={"Allele":"allele", "Peptide":"seq"}, inplace=True)
mhcma["label"]=1
mhcma["dataset"]="MHCMA"
mhcma["method_group"]="MS"
mhcma=standardize_allele_format(mhcma, no_hla_prefix=True)

In [ ]:
print_data_stats(mhcma)
print("mhcma", mhcma.allele.nunique())

Total: 442,691
Pos: 442,691 | 100.00%
Neg: 0 | 0.00%
N alleles: 123
mhcma 123


#### HLAthena

 - Collect data from supplementary table 1 (Sup_Tab1) from https://www.nature.com/articles/s41587-019-0322-9. 

In [ ]:
%%time
# pub 2; sup table 1
filt_pep_list=pd.read_excel(f"{DATA_DIR}/Training_Datasets/Collected/HLAthena/NIHMS1541512-supplement-Sup_Tab1.xlsx", sheet_name="e. Filtered peptide list")

CPU times: user 6.01 s, sys: 51.3 ms, total: 6.06 s
Wall time: 6.06 s


In [ ]:
# format
hlathena_data=filt_pep_list[["Allele", "Length", "Peptide"]]
hlathena_data.columns=["allele", "seq_len", "seq"]
hlathena_data=standardize_allele_format(hlathena_data)
hlathena_data["label"]=1
hlathena_data["dataset"]="HLAthena"
# hlathena_data=hlathena_data.query("seq_len>=8 and seq_len <=14")
hlathena_data.drop_duplicates(subset=["seq", "allele", "label"], inplace=True)
hlathena_data=map_allele_sequences_efficient_alt(hlathena_data, mhc_seq_dict)
hlathena_data["method_group"]="MS"
hlathena_data["sa_ma"]="SA"

In [ ]:
print_data_stats(hlathena_data)
print("N alleles", hlathena_data.allele.nunique())

Total: 184,183
Pos: 184,183 | 100.00%
Neg: 0 | 0.00%
N alleles: 92
N alleles 92


#### NetMHCpan4.1 EL and BA data

 - Downloaded `NetMHCpan_train.tar.gz` from https://services.healthtech.dtu.dk/suppl/immunology/NAR_NetMHCpan_NetMHCIIpan/.

In [ ]:
# read allele list
import re

def parse_file_to_dict(file_path):
   result = {}
   
   with open(file_path, 'r') as f:
       for line in f:
           line = line.strip()
           if not line:
               continue
           parts = re.split(r'\s+', line, 1)
           if len(parts) == 2:
               key = parts[0]
               values = [val.strip() for val in parts[1].split(',')]
               result[key] = values
   
   return result

# get allelelist
nmp_allelelist = parse_file_to_dict(f'{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41/allelelist')

# isolate single alleles and hla alleles
nmp_allelelist_sa = {k: v[0] for k, v in nmp_allelelist.items() if len(v) == 1}

nmp_allelelist_sa_hla = {k: v for k, v in nmp_allelelist_sa.items() if k.startswith('HLA')}

# isolate non hla alleles 
nmp_allelelist_sa_non_hla={k: v for k, v in nmp_allelelist_sa.items() if not k.startswith('HLA')}

# read in pseudosequences
mhc_pseudosequences_nmp = pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41/MHC_pseudo.dat", sep=r"\s+", engine="python", header=None)
mhc_pseudosequences_nmp.columns=["allele", "seq"]

In [ ]:
# get raw data from their folder of info
nmp_files=os.listdir(f"{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41")

# ba data
nmp_ba_data=pd.DataFrame()
for f in tqdm(nmp_files):
    if f.endswith("ba"):
        df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41/{f}", sep=" ", header=None)
        nmp_ba_data=pd.concat([nmp_ba_data, df], axis=0)

nmp_ba_data.columns=["seq", "ba_value", "allele"]

# el data
nmp_el_data=pd.DataFrame()
for f in tqdm(nmp_files):
    if f.endswith("el"):
        df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/NetMHCpan41/{f}", sep=" ", header=None)
        nmp_el_data=pd.concat([nmp_el_data, df], axis=0)

nmp_el_data.columns=["seq", "label", "allele"]

  0%|          | 0/12 [00:00<?, ?it/s]

100%|██████████| 12/12 [00:05<00:00,  2.17it/s]


In [ ]:
# get all el sa human data
nmp_el_data_hla=nmp_el_data[nmp_el_data.allele.isin(list(nmp_allelelist_sa_hla.values()))]

# format allele
nmp_el_data_hla.allele=nmp_el_data_hla.allele.str.replace("HLA-", "").str.replace(":", "")

print_data_stats(nmp_el_data_hla)

Total: 3,679,405
Pos: 197,547 | 5.37%
Neg: 3,481,858 | 94.63%
N alleles: 130


In [ ]:
# process ba data

# convert BA data to binary, add to the training set (use the nmp thresholds, above 0.4260 is binding and below is non-binding) (BA values -> 1 - log(IC50)/log(50000))
nmp_ba_data_labels=[1 if v >= 0.4260 else 0 for v in nmp_ba_data.ba_value.tolist()]
nmp_ba_data["label"]=nmp_ba_data_labels
 
# apply length filter
nmp_ba_data["seq_len"]=nmp_ba_data.seq.apply(len)
nmp_ba_data=nmp_ba_data.query("seq_len >= 8 and seq_len <= 14")
# print(nmp_ba_data.label.value_counts())
# print_data_stats(nmp_ba_data)

# get hla data only
nmp_ba_data_hla=nmp_ba_data[nmp_ba_data.allele.str.startswith("HLA-")]

# format allele
nmp_ba_data_hla.allele=nmp_ba_data_hla.allele.str.removeprefix("HLA-").str.replace(":","")

# remove hla e and g
nmp_ba_data_hla=nmp_ba_data_hla[~nmp_ba_data_hla.allele.isin(["E0101", "E0103"])]

# drop duplicates
nmp_ba_data_hla.drop_duplicates(subset=["seq", "allele", "label"], inplace=True)

# print stats
print_data_stats(nmp_ba_data_hla)

Total: 170,107
Pos: 42,001 | 24.69%
Neg: 128,106 | 75.31%
N alleles: 109


In [ ]:
"""
Save BA data for FT
"""
# nmp_ba_data_hla.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/nmp_ba_data_hla.csv")

'\nSave BA data for FT\n'

#### HLApollo

 - Data downloaded from Zenodo (Version v1): https://zenodo.org/records/7951718. Use specifically BM1 for training.

In [ ]:
%%time
# bm1 test dat; train is more expansive, need to get all data files
bm1_test=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/HLApollo/BM1/test/BM1_test.csv")

# add sa ma col and process
bm1_test["sa_ma"]=["MA" if "_" in a else "SA" for a in bm1_test.allele]
bm1_test.rename(columns={"peptide": "seq", "el": "label"}, inplace=True)

bm1_test_sa=bm1_test.query("sa_ma=='SA'")
bm1_test_sa.allele=bm1_test_sa.allele.str.replace("*", "").str.replace(":", "")

print_data_stats(bm1_test_sa)
print(bm1_test_sa.allele.nunique())

Total: 1,412,200
Pos: 14,122 | 1.00%
Neg: 1,398,078 | 99.00%
N alleles: 107
107
CPU times: user 8.65 s, sys: 1.18 s, total: 9.82 s
Wall time: 9.83 s


In [ ]:
%%time
# get pos train set
bm1_train_pos=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/HLApollo/BM1/train/BM1_positive_train.csv")

# label sa and ma
bm1_train_pos["sa_ma"]=["MA" if "_" in a else "SA" for a in bm1_train_pos.allele]

CPU times: user 2.25 s, sys: 132 ms, total: 2.39 s
Wall time: 2.39 s


In [ ]:
%%time
# import neg train set
# there are multiple, so we will have to aggregate later -> see formatting here
bm1_train_neg=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/HLApollo/BM1/train/BM1_negative_train_set_01.csv")
bm1_train_neg["sa_ma"]=["MA" if "_" in a else "SA" for a in bm1_train_neg.allele]

CPU times: user 1.74 s, sys: 116 ms, total: 1.85 s
Wall time: 1.85 s


In [ ]:
# train files
train_files=os.listdir(f"{DATA_DIR}/Training_Datasets/Collected/HLApollo/BM1/train/")
print(len(train_files), train_files)

21 ['BM1_negative_train_set_14.csv', 'BM1_negative_train_set_18.csv', 'BM1_negative_train_set_11.csv', 'BM1_negative_train_set_04.csv', 'BM1_negative_train_set_15.csv', 'BM1_negative_train_set_16.csv', 'BM1_positive_train.csv', 'BM1_negative_train_set_07.csv', 'BM1_negative_train_set_01.csv', 'BM1_negative_train_set_20.csv', 'BM1_negative_train_set_12.csv', 'BM1_negative_train_set_09.csv', 'BM1_negative_train_set_05.csv', 'BM1_negative_train_set_08.csv', 'BM1_negative_train_set_10.csv', 'BM1_negative_train_set_13.csv', 'BM1_negative_train_set_03.csv', 'BM1_negative_train_set_02.csv', 'BM1_negative_train_set_06.csv', 'BM1_negative_train_set_17.csv', 'BM1_negative_train_set_19.csv']


In [ ]:
%%time
# aggregate all train
bm1_train_dfs={}
for f in tqdm(train_files):
    if f!="BM1_train_processed.csv":
        bm1_train_df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/HLApollo/BM1/train/{f}")
        bm1_train_df["filename"]=f
        bm1_train_df["sa_ma"]=["MA" if "_" in a else "SA" for a in bm1_train_df.allele]
        bm1_train_dfs[f]=bm1_train_df
bm1_train=pd.concat(bm1_train_dfs.values())

  0%|          | 0/21 [00:00<?, ?it/s]

100%|██████████| 21/21 [00:40<00:00,  1.91s/it]


CPU times: user 39.1 s, sys: 4.04 s, total: 43.1 s
Wall time: 43.1 s


In [ ]:
# process
bm1_train.rename(columns={"peptide": "seq", "el": "label"}, inplace=True)

bm1_train_sa=bm1_train.query("sa_ma=='SA'")
bm1_train_sa.allele=bm1_train_sa.allele.str.replace("*", "").str.replace(":", "")

print_data_stats(bm1_train_sa)
print(bm1_train_sa.allele.nunique())

Total: 4,027,443
Pos: 191,783 | 4.76%
Neg: 3,835,660 | 95.24%
N alleles: 135
135


#### MHCFlurry2.0

 - Collect supplementary tables from https://data.mendeley.com/datasets/zx3kjzc3yx/3. Specifically, `Data_S2.csv.gz` and `Data_S3.csv.gz`.

In [ ]:
os.listdir(f"{DATA_DIR}/Training_Datasets/Collected/MHCFlurry")
s2=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/MHCFlurry/Data_S2.csv.gz", compression='gzip') # monoallelic
s3=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/MHCFlurry/Data_S3.csv")

In [ ]:
# format monoallelic data
# contains flank data, but reduce length to 10 only, or void for now.
s2.rename(columns={"peptide":"seq", "hit":"label", "hla":"allele"}, inplace=True)
s2=standardize_allele_format(s2, no_hla_prefix=False)
s2["dataset"]="MHCFlurry"
s2["method_group"]="MS"

# restrict length to 14

In [ ]:
# format s3 (mix of ms and affinity)
s3.rename(columns={"peptide":"seq"}, inplace=True)
s3=standardize_allele_format(s3, no_hla_prefix=False)
s3["dataset"]="MHCFlurry"
s3_ms=s3.query("measurement_kind=='mass_spec'")
s3_ms["method_group"]="MS" 

In [ ]:
# get non ms data
s3_ba=s3.query("measurement_kind!='mass_spec'")
s3_ba["method_group"]="nonMS"

In [ ]:
# map inequality to labels
# for measurement inequalities, the < indicates the actual value is less than the measured value, so peptides are known to be binders, and the opposite for >.
inequality_label_map={"<":1, ">":0}
s3_ms["label"]=s3_ms.measurement_inequality.map(inequality_label_map)

In [ ]:
"""
Merged exact and non-exact values, for now, just use a threshold of 500 nM, anything <= considered a binder, and anything greater is considered a non-binder for binary classification.

Alternatively, drop peptides that have inequality > in 500 nM or less and peptides that are < in 500 nM or more group.
"""

# transformed values
s3_ba["transformed_score"]=1-(np.log(s3_ba.measurement_value)/np.log(5000))

# classify nosiey vs exact
s3_ba_exact=s3_ba.query("measurement_inequality == '='")
s3_ba_noisy=s3_ba.query("measurement_inequality != '='")

# compute labels
labels=[1 if mv <= 500 else 0 for mv in s3_ba_exact.measurement_value]
s3_ba_exact["label"]=labels

labels=[1 if mv <= 500 else 0 for mv in s3_ba_noisy.measurement_value]
s3_ba_noisy["label"]=labels

# repeat for whole dataset
labels=[1 if mv <= 500 else 0 for mv in s3_ba.measurement_value]
s3_ba["label"]=labels

# sns.kdeplot(s3_ba.query("label==1").transformed_score, fill=True, color="mediumblue", alpha=0.6)
# sns.kdeplot(s3_ba.query("label==0").transformed_score, fill=True, color="red", alpha=0.6)
# plt.show()

In [ ]:
# merge s2 and s3_ms for stats
s2_s3_ms=pd.concat([s2, s3_ms])
s2_s3_ms.drop_duplicates(subset=["seq", "allele", "label"], inplace=True)

In [ ]:
print_data_stats(s2)
print("N alleles", s2.allele.nunique())

Total: 13,271,500
Pos: 132,715 | 1.00%
Neg: 13,138,785 | 99.00%
N alleles: 92
N alleles 92


In [ ]:
print_data_stats(s3_ms)
print("N alleles", s3_ms.allele.nunique())

Total: 474,337
Pos: 456,824 | 96.31%
Neg: 17,513 | 3.69%
N alleles: 144
N alleles 144


In [ ]:
print_data_stats(s2_s3_ms)
print("N alleles", s2_s3_ms.allele.nunique())

Total: 13,532,272
Pos: 456,818 | 3.38%
Neg: 13,075,454 | 96.62%
N alleles: 144
N alleles 144


#### Ligand.MHC Atlas data

 - From http://modinfor.com/Ligand.MHC-Atlas/Download.php.

In [ ]:
"""
# NOTE: original, but if this errors on clusters, run below.
# get hla names for cancer and benchmark sets
url = "http://modinfor.com/Ligand.MHC-Atlas/Download.php"
base = "http://modinfor.com/Ligand.MHC-Atlas/"

resp = requests.get(url, timeout=30)
resp.raise_for_status()
html = resp.text

# hla class i download names
hla_matches = re.findall(r'Download/(HLA-[ABCEG][A-Za-z0-9_-]+)\.zip', html)
hla_names = sorted(set(hla_matches))

print(f"Found {len(hla_names)} HLA entries")

hla_df = pd.DataFrame({"name": hla_names})
hla_df["url"] = base + "Download/" + hla_df["name"] + ".zip"
hla_df["type"] = "hla

"""

'\n# NOTE: original, but if this errors on clusters, run below.\n# get hla names for cancer and benchmark sets\nurl = "http://modinfor.com/Ligand.MHC-Atlas/Download.php"\nbase = "http://modinfor.com/Ligand.MHC-Atlas/"\n\nresp = requests.get(url, timeout=30)\nresp.raise_for_status()\nhtml = resp.text\n\n# hla class i download names\nhla_matches = re.findall(r\'Download/(HLA-[ABCEG][A-Za-z0-9_-]+)\\.zip\', html)\nhla_names = sorted(set(hla_matches))\n\nprint(f"Found {len(hla_names)} HLA entries")\n\nhla_df = pd.DataFrame({"name": hla_names})\nhla_df["url"] = base + "Download/" + hla_df["name"] + ".zip"\nhla_df["type"] = "hla\n\n'

In [ ]:
import time
import requests

url = "http://modinfor.com/Ligand.MHC-Atlas/Download.php"

headers = {
    "User-Agent": "Mozilla/5.0"
}

last_err = None
for attempt in range(5):
    try:
        resp = requests.get(
            url,
            headers=headers,
            timeout=(10, 120),  # connect timeout, read timeout
        )
        resp.raise_for_status()
        html = resp.text
        break
    except requests.RequestException as e:
        last_err = e
        print(f"Attempt {attempt + 1}/5 failed: {e}")
        time.sleep(5 * (attempt + 1))
else:
    raise RuntimeError(f"Could not fetch Ligand.MHC-Atlas download page after retries: {last_err}")

hla_matches = re.findall(r'Download/(HLA-[ABCEG][A-Za-z0-9_-]+)\.zip', html)
hla_names = sorted(set(hla_matches))

print(f"Found {len(hla_names)} HLA entries")

hla_df = pd.DataFrame({"name": hla_names})
hla_df["url"] = base + "Download/" + hla_df["name"] + ".zip"
hla_df["type"] = "hla"

Found 152 HLA entries


In [ ]:
%%time
# download all files
ligand_mhc_atlas_atlas_cancer_df, fail_cancer_df = download_concat_ligand_mhc_atlas(
    hla_df=hla_df,
    dataset_type="cancer",
    name_col="name",
    ignore_errors=True,
    return_failures=True,
)

CPU times: user 24.2 s, sys: 4.17 s, total: 28.4 s
Wall time: 5min 15s


In [ ]:
# format
ligand_mhc_atlas_atlas_cancer_df.rename(columns={"HLA": "allele", "Immunopeptide": "seq"}, inplace=True)
ligand_mhc_atlas_atlas_cancer_df.allele=ligand_mhc_atlas_atlas_cancer_df.allele.str.removeprefix("HLA-").str.replace("*", "").str.replace(":", "")
ligand_mhc_atlas_atlas_cancer_df_proccessed=full_process_data(ligand_mhc_atlas_atlas_cancer_df, mhc_seq_dict, remove_flank=False, add_pseudo_label=False)
ligand_mhc_atlas_atlas_cancer_df_proccessed=create_seq_allele_column(ligand_mhc_atlas_atlas_cancer_df_proccessed)
ligand_mhc_atlas_atlas_cancer_df_proccessed["label"]=1

In [ ]:
%%time
"""
Save base df
"""
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
# ligand_mhc_atlas_atlas_cancer_df_processed.to_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_atlas_cancer_df_processed.csv")

CPU times: user 42 µs, sys: 17 µs, total: 59 µs
Wall time: 79.2 µs


#### Pathogen datasets

__Pathogen studies__

 - CoV set: https://www.cell.com/cell/fulltext/S0092-8674(21)00701-7
 - Zika set: https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0335726#sec008
 - Dengue set: NA
 - Mtb set: https://pmc.ncbi.nlm.nih.gov/articles/PMC10159623/#s7


##### CoV

In [ ]:
# get cov set and process
# os.listdir("CoV_MS_Data")

cov_hek293=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/CoV_MS_Data/cov_set_hek293t.csv")
cov_a549=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Collected/CoV_MS_Data/cov_set_a549.csv")

In [ ]:
# format
cov_hek293.rename(columns={"Sequence": "seq"}, inplace=True)
cov_a549.rename(columns={"Sequence": "seq"}, inplace=True)

cov_hek293["cell_type"]="hek293"
cov_a549["cell_type"]="a549"

cov_hek293.reset_index(inplace=True, drop=True)
cov_a549.reset_index(inplace=True, drop=True)

# cov_hek293["peptide_id"]=cov_hek293.index.tolist()
# cov_a549["peptide_id"]=cov_a549.index.tolist()

cov_hek293.dropna(subset=["seq"], inplace=True)
cov_a549.dropna(subset=["seq"], inplace=True)

cov_hek293.seq=cov_hek293.seq.str.replace("*", "")
cov_hek293.seq=cov_hek293.seq.str.replace(" ", "")

cov_a549.seq=cov_a549.seq.str.replace("*", "")
cov_a549.seq=cov_a549.seq.str.replace(" ", "")

In [ ]:
print(cov_a549.shape)
print(cov_hek293.shape)

(21, 18)
(20, 18)


In [ ]:
# merge first then create id
cov_hek293_a549=pd.concat([cov_hek293, cov_a549])
cov_hek293_a549.reset_index(inplace=True, drop=True)
cov_hek293_a549["peptide_id"]=cov_hek293_a549.index.tolist()
print(cov_hek293_a549.shape)

(41, 20)


In [ ]:
# expand
donor_dict={
    "hek293": ["A0201", "B0702", "C0702"],
    "a549": ["A2501", "A3001", "B1801", "B4403", "C1203", "C1601"],
}

# cov_hek293_expanded=expand_alleles_by_donor(cov_hek293, "seq", "cell_type", donor_dict)
# cov_a549_expanded=expand_alleles_by_donor(cov_a549, "seq", "cell_type", donor_dict)
cov_hek293_a549_expanded=expand_alleles_by_donor(cov_hek293_a549, "seq", "cell_type", donor_dict)

ms_cov_set=cov_hek293_a549_expanded.copy()

Expanding donors: 100%|██████████| 2/2 [00:00<00:00, 476.22it/s]


##### Zika

In [ ]:
zikv_ms_hla_i_peptides = [
    "KEAMEIIKKF",
    "RLVDPINVV",
    "ALWDVPAPKEV",
    "YLQDGLIASL",
    "RMLLDNIYL",
    "VLPEIVREA",
    "LQDGLIASL",
    "RLLGSTQVGV",
    "AEMEEALRGL",
    "TEVEVPERAW",
    "EEALRGLPVRY",
    "ILDGERVIL",
    "NQMAIIIMV",
    "YLIPGLQAA",
    "YLAGASLIYTV",
    "TMMETLERL",
    "SLINGVVRL",
    "EEVPFCSHHF",
    "QEWKPSTGW",
]

ms_zika_set=pd.DataFrame({"seq": zikv_ms_hla_i_peptides})
ms_zika_set["cell_line"]="usp7"
ms_zika_set.reset_index(inplace=True, drop=True)
ms_zika_set["peptide_id"]=ms_zika_set.index.tolist()
print(ms_zika_set.shape)

donor_dict={
    "usp7": ["A0201", "B4402", "C0501"]
}

ms_zika_set=expand_alleles_by_donor(ms_zika_set, "seq", "cell_line", donor_dict)
ms_zika_set=create_seq_allele_column(ms_zika_set)
print(ms_zika_set.shape)

(19, 3)


Expanding donors: 100%|██████████| 1/1 [00:00<00:00, 560.81it/s]

(57, 5)


##### Mtb

In [ ]:
# mtb peptides
leddy_mtb_validated_hla_i_peptides = [
    "VYLTAHNAL",
    "EAVQDVARTY",
    "EVHSAMLNY",
    "AEILRGVSA",
    "AEHGMPGVP",
    "LPFEDAPLI",
    "AAWGGSGSEAY",
    # "ELDEISTNIRQAGVQY", # TOO LONG!
    "IDHQFVATL",
    "NASPVAQSY",
    "VPLEGGGRL",
    "TQHDAADALF",
    "SEAYQGVQQKW",
    "QEQASQQIL",
    "LLDEGKQSL",
    "QTVEDEARRMW",
]

# organize based on donors

leddy_mtb_validated_hla_i_peptides = {

    # donor b
    "B": ["LLDEGKQSL"],

    # donor c
    "C": ["SEAYQGVQQKW",
    "QTVEDEARRMW",
    "LLDEGKQSL"],

    # donor d
    "D": ["VYLTAHNAL",
    "EVHSAMLNY",
    "EAVQDVARTY",
    "AEHGMPGVP",
    "AEILRGVSA"],

    # donor e
    "E": ["VYLTAHNAL",
    "LPFEDAPLI"],
    
    # donor f
    "F": ["TQHDAADALF",
    # "ELDEISTNIRQAGVQY",
    "VPLEGGGRL",
    "AAWGGSGSEAY",
    "NASPVAQSY",
    "EAVQDVARTY"],
}

def donor_dict_to_df(d):
    return pd.DataFrame([
        {"donor": donor, "seq": seq}
        for donor, seqs in d.items()
        for seq in seqs
    ])

ms_mtb_set=donor_dict_to_df(leddy_mtb_validated_hla_i_peptides)
ms_mtb_set.reset_index(inplace=True, drop=True)
ms_mtb_set["peptide_id"]=ms_mtb_set.index.tolist()
print(ms_mtb_set.shape)

donor_dict={
    "B": ["A0201", "A2402", "B0702", "B2707", "C0702", "C1502"],
    "C": ["A0101", "A0201", "B4403", "B5701", "C0602", "C1601"],
    "D": ["A2301", "A2601", "B3501", "B5001", "C0401", "C0602"],
    "E": ["A1101", "A2402", "B3906", "B5101", "C0702", "C1502"],
    "F": ["A0101", "A2601", "B3501", "B3801", "C0401", "C1203"],
}

ms_mtb_set=expand_alleles_by_donor(ms_mtb_set, "seq", "donor", donor_dict)
ms_mtb_set=create_seq_allele_column(ms_mtb_set)
print(ms_mtb_set.shape)

(16, 3)


Expanding donors:   0%|          | 0/5 [00:00<?, ?it/s]

Expanding donors: 100%|██████████| 5/5 [00:00<00:00, 752.04it/s]

(96, 5)


In [ ]:
"""
Save MS pathogen sets
"""
# ms_cov_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Pathogen_Sets/ms_cov_set.csv")
# ms_zika_set.to_csv(f"{DATA_DIR}Training_Datasets/Processed/MS_Train_Test_Sets/Pathogen_Sets/ms_zika_set.csv")
# ms_mtb_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Pathogen_Sets/ms_mtb_set.csv")

### Non-MS datasets

 - Note: BA dataset was retrieved above.

#### Yeast display

 - Data from: https://www.pnas.org/doi/10.1073/pnas.2514741122. Collected from Zenodo: https://zenodo.org/records/17093099.

In [ ]:
# files
os.listdir(f"{DATA_DIR}/Training_Datasets/Collected/Holec_et_al_2025")

['all_pathogen_positives.xlsx',
 'pathogen_library_ranks.xlsx',
 'pathogen_library_datasets_by_gene']

In [ ]:
# get ranks library sheet names
pathogen_library_ranks_excel = pd.ExcelFile(f"{DATA_DIR}/Training_Datasets/Collected/Holec_et_al_2025/pathogen_library_ranks.xlsx")
print(pathogen_library_ranks_excel.sheet_names)

['SARS-CoV-2_N-gene', 'SARS-CoV-2_S-gene', 'Dengue', 'Mtb-T7SS (PE_PGRS16)', 'Mtb-T7SS (PPE1)', 'Mtb-T7SS (PPE3)', 'Mtb-T7SS (PPE10)', 'Mtb-T7SS (PPE20)', 'Mtb-T7SS (PPE29)', 'Mtb-T7SS (PPE24)', 'Mtb-T7SS (PPE36)', 'Mtb-T7SS (PPE40)', 'Mtb-T7SS (PPE34)', 'Mtb-T7SS (PPE45)', 'Mtb-T7SS (PPE49)', 'Mtb-T7SS (PPE47_ppe48)', 'Mtb-T7SS (PPE51)', 'Mtb-T7SS (PPE54)', 'Mtb-T7SS (PPE66)', 'Mtb-T7SS (PPE62)', 'Mtb-T7SS (PPE63)', 'Mtb-T7SS (PPE69)', 'Mtb-T7SS (PPE17)', 'Mtb-T7SS (PE_PGRS3)', 'Mtb-T7SS (PPE21)', 'Mtb-T7SS (PPE22)', 'Mtb-T7SS (PPE23)', 'Mtb-T7SS (PPE38)', 'Mtb-T7SS (PPE33)', 'Mtb-T7SS (PPE42)', 'Mtb-T7SS (PPE44)', 'Mtb-T7SS (PPE50)', 'Mtb-T7SS (PPE59)', 'Mtb-T7SS (PPE61)', 'Mtb-T7SS (PPE68)', 'Mtb-T7SS (PPE65)', 'Mtb-T7SS (PPE13)', 'Mtb-T7SS (PPE14)', 'Mtb-T7SS (PPE15)', 'Mtb-T7SS (PPE16)', 'Mtb-T7SS (PPE19)', 'Mtb-T7SS (PE_PGRS47)', 'Mtb-T7SS (PE15)', 'Mtb-T7SS (PE16)', 'Mtb-T7SS (PPE11)', 'Mtb-T7SS (PPE2)', 'Mtb-T7SS (PPE4)', 'Mtb-T7SS (PPE26)', 'Mtb-T7SS (PPE28)', 'Mtb-T7SS (PPE25

In [ ]:
# get covid gene pathogen ranks
plr_cov_n_gene = pd.read_excel(f"{DATA_DIR}/Training_Datasets/Collected/Holec_et_al_2025/pathogen_library_ranks.xlsx", sheet_name='SARS-CoV-2_N-gene', index_col=0)
plr_cov_s_gene = pd.read_excel(f"{DATA_DIR}/Training_Datasets/Collected/Holec_et_al_2025/pathogen_library_ranks.xlsx", sheet_name='SARS-CoV-2_S-gene', index_col=0)

In [ ]:
# get all positives
all_pathogen_positives=pd.read_excel(f"{DATA_DIR}/Training_Datasets/Collected/Holec_et_al_2025/all_pathogen_positives.xlsx", index_col=0)

In [ ]:
%%time
"""
How to get the data. 

The pathogen_library_datasets_by_gene contains each protein set above from the all pathogen positives object followed by binary or prob. We want the binary ones, and we need to extract all the genes per table, where the genes are the sheet. This will give us positives and negatives. 
"""
# read in all pathogen binary data
path_to_pldbg=f"{DATA_DIR}/Training_Datasets/Collected/Holec_et_al_2025/pathogen_library_datasets_by_gene/"

holec_et_al_pathogen_data=process_binary_excel_files(path_to_pldbg)

# check that positives match
assert len(holec_et_al_pathogen_data.query("label == 1")) == len(all_pathogen_positives)

holec_et_al_pathogen_data=holec_et_al_pathogen_data.drop_duplicates(subset=["seq", "allele", "label"])
print_data_stats(holec_et_al_pathogen_data)

Processing files:   0%|          | 0/16 [00:00<?, ?it/s]

Processing files: 100%|██████████| 16/16 [00:17<00:00,  1.07s/it]


Combining all data...
Total: 1,552,352
Pos: 28,503 | 1.84%
Neg: 1,523,849 | 98.16%
N alleles: 16
CPU times: user 17.5 s, sys: 412 ms, total: 17.9 s
Wall time: 17.9 s


In [ ]:
"""
Save processed data
"""
# holec_et_al_pathogen_data.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/Holec_et_al_2025/holec_et_al_pathogen_data.csv")

'\nSave processed data\n'

#### EpiScan

 - Data from https://www.nature.com/articles/s41587-022-01566-x. See source data.
 - SD2: Random 9-mer (R9) library.
 - SD3: SARS-Cov-2 screen.
 - SD6: Retraining (RT) library.

In [ ]:
%%time
"""
Get SD2 (random 9-mer library).
"""
os.listdir(f"{DATA_DIR}/Training_Datasets/Collected/EpiScan/")
sd2 = pd.read_excel(f"{DATA_DIR}/Training_Datasets/Collected/EpiScan/SD2.xlsx", engine="openpyxl", header=1) # random 9-mers, in sd7, prefix rand9 (n=6)

CPU times: user 12.5 s, sys: 163 ms, total: 12.7 s
Wall time: 12.7 s


In [ ]:
"""
Parse SD2:
 - Iterate through allele binders and subtract the random 9-mers -> those become negatives for each allele specificially. 
 - Do not use abacavir binders at the moment.
 - The SPP KO for A0201 showed better sensitivity, therefore, use it over the regular version.
"""

sd2_binder_column_map={"A2 SPP KO binders": "A0201", "A2 binders": "A0201", "A3 binders": "A0301", "B8 binders": "B0801", "B57 binders": "B5701"}

# isolate all random 9mers
sd2_all_random_9mers=sd2[["all random 9mers"]]
sd2_all_random_9mers.rename(columns={"all random 9mers": "seq"}, inplace=True)
sd2_all_random_9mers["label"]=0

sd2_formatted=pd.DataFrame()

for b,a in tqdm(sd2_binder_column_map.items()):
    sd2_f=sd2[[b]].dropna()
    sd2_f["label"]=1
    sd2_f["allele"]=a
    sd2_f.rename(columns={b:"seq"}, inplace=True)
    sd2_f_negs=sd2_all_random_9mers[~sd2_all_random_9mers.seq.isin(list(sd2_f.seq))]
    sd2_f_negs["allele"]=a
    sd2_f=pd.concat([sd2_f, sd2_f_negs])

    # add thing for spp ko binders
    if "SPP" in b:
        sd2_f["spp_ko"]=1
    else:
        sd2_f["spp_ko"]=0

    sd2_formatted=pd.concat([sd2_formatted, sd2_f])

# create seq allele column
sd2_formatted=create_seq_allele_column(sd2_formatted)

# filter sd2 for spp ko a0201 for that allele
sd2_formatted_spp_ko = sd2_formatted[~((sd2_formatted.allele == 'A0201') & (sd2_formatted.spp_ko == 0))]

sd2_formatted_spp_ko.drop_duplicates(subset=["seq", "allele", "label"], inplace=True)

# print stats
print_data_stats(sd2_formatted_spp_ko)

100%|██████████| 5/5 [00:00<00:00, 10.78it/s]


Total: 1,919,616
Pos: 4,218 | 0.22%
Neg: 1,915,398 | 99.78%
N alleles: 4


In [ ]:
%%time
"""
Import SD6 binders for RT ESP set.
"""
def process_episcan_sd6_df(df, allele=None):
    dfs = []
    
    for col, label in [('Decoys', 0), ('Binders - low threshold', 1), ('Binders - high threshold', 2)]:
        temp_df = df[[col]].copy()
        temp_df = temp_df.dropna()
        temp_df.columns = ['seq']
        temp_df['label'] = label
        dfs.append(temp_df)
    
    result_df = pd.concat(dfs, ignore_index=True)
    
    if allele:
        result_df['allele'] = allele
    
    return result_df

sd6_allele_map={"A2":"A0201", "A3":"A0301", "B8":"B0801", "B57":"B5701"}

sd6_dfs={}
for sheet, allele in tqdm(sd6_allele_map.items()):
    df = pd.read_excel(f"{DATA_DIR}/Training_Datasets/Collected/EpiScan/SD6.xlsx", sheet_name=sheet)
    sd6_dfs[allele]=df

# format each sheet (0: decoys, 1: low threshold binders, 2: high threshold binders)
for allele, df in tqdm(sd6_dfs.items()):
    df_f=process_episcan_sd6_df(df, allele=allele)
    sd6_dfs[allele]=df_f

sd6_df = pd.concat(sd6_dfs.values(), ignore_index=True)
sd6_df["original_label"]=sd6_df.label.tolist() # save original label
sd6_df.label=sd6_df.label.replace(2, 1)
sd6_df.drop_duplicates(subset=["seq", "allele", "label"], inplace=True)
sd6_df=create_seq_allele_column(sd6_df)
print_data_stats(sd6_df, allele_dist=True)

100%|██████████| 4/4 [00:00<00:00, 35.11it/s]


Total: 729,673
Pos: 118,543 | 16.25%
Neg: 611,130 | 83.75%
N alleles: 4
A0201
  - Total: 195,393
  - Pos: 30,093 | 15.40%
  - Neg: 165,300 | 84.60%
A0301
  - Total: 179,192
  - Pos: 68,444 | 38.20%
  - Neg: 110,748 | 61.80%
B0801
  - Total: 182,680
  - Pos: 8,359 | 4.58%
  - Neg: 174,321 | 95.42%
B5701
  - Total: 172,408
  - Pos: 11,647 | 6.76%
  - Neg: 160,761 | 93.24%
CPU times: user 35.7 s, sys: 231 ms, total: 36 s
Wall time: 36 s


In [ ]:
%%time
"""
SD3 CoV data
"""
sd3 = pd.read_excel(f"{DATA_DIR}/Training_Datasets/Collected/EpiScan/SD3.xlsx", engine="openpyxl", header=1) # cov screen peptides, in sd7, prefix CoV2 (n=12)

sd3_binder_column_map={"A1 binders": "A0101", "A2 SPP WT binders": "A0201", "A3 binders": "A0301", "B8 binders": "B0801", "A24:02 binders": "A2402", "B07:02 binders": "B0702", "B40 binders":"B4001", "B51 binders": "B5101", "C4 binders": "C0401", "C07:01 binders": "C0701", "C07:02 binders": "C0702"}

# isolate all peptides
sd3_all_peptides=sd3[["all peptides"]]
sd3_all_peptides.rename(columns={"all peptides": "seq"}, inplace=True)
sd3_all_peptides["label"]=0

sd3_formatted=pd.DataFrame()

for b,a in tqdm(sd3_binder_column_map.items()):
    sd3_f=sd3[[b]].dropna()
    sd3_f["label"]=1
    sd3_f["allele"]=a
    sd3_f.rename(columns={b:"seq"}, inplace=True)
    sd3_f_negs=sd3_all_peptides[~sd3_all_peptides.seq.isin(list(sd3_f.seq))]
    sd3_f_negs["allele"]=a
    sd3_f=pd.concat([sd3_f, sd3_f_negs])
    sd3_formatted=pd.concat([sd3_formatted, sd3_f])

print_data_stats(sd3_formatted)

100%|██████████| 11/11 [00:00<00:00, 121.65it/s]

Total: 326,579
Pos: 767 | 0.23%
Neg: 325,812 | 99.77%
N alleles: 11
CPU times: user 926 ms, sys: 11.7 ms, total: 937 ms
Wall time: 935 ms


In [ ]:
%%time
"""
Save EpiScan raw datasets
"""

episcan_train_test_dir=f"{DATA_DIR}/Training_Datasets/Processed/EpiScan"
os.makedirs(episcan_train_test_dir, exist_ok=True)

# sd2_formatted_spp_ko.to_csv(f"{episcan_train_test_dir}/sd2_formatted_spp_ko.csv")
# sd6_df.to_csv(f"{episcan_train_test_dir}/sd6_df.csv")

CPU times: user 4.1 s, sys: 252 ms, total: 4.35 s
Wall time: 4.35 s


#### ESCAPE-seq

 - Download source data from: https://www.nature.com/articles/s41588-025-02268-1. For reported neoantigen sets, download supplementary table 7.

In [ ]:
"""
Get oncogene data
"""
oncogene_mut_df = pd.read_excel(f'{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq/escape_seq_source_data.xls', sheet_name='oncogene')
oncogene_wt_df = pd.read_excel(f'{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq/escape_seq_source_data.xls', sheet_name='onco_wt')

In [ ]:
# flatten
def flatten_allele_df(df, allele_pattern=r'^[ABC]\*\d+:\d+$', fixed_cols=None):
    # auto detect allele cols and fixed cols
    allele_cols = [c for c in df.columns if re.match(allele_pattern, c)]
    
    if fixed_cols is None:
        fixed_cols = [c for c in df.columns if c not in allele_cols]
    
    melted = df.melt(
        id_vars=fixed_cols,
        value_vars=allele_cols,
        var_name='allele',
        value_name='score'
    )
    
    return melted.reset_index(drop=True)

oncogene_mut_df_flat=flatten_allele_df(oncogene_mut_df, allele_pattern=r'^[ABC]\*\d+:\d+$', fixed_cols=None)

oncogene_wt_df_flat=flatten_allele_df(oncogene_wt_df, allele_pattern=r'^[ABC]\*\d+:\d+$', fixed_cols=None)

In [ ]:
# clean df
def clean_flattened_df(df, score_col_name, allele_col='allele', aa_col='AA', score_col='score'):
    result = df.copy()
    result[allele_col] = result[allele_col].str.replace('*', '', regex=False).str.replace(':', '', regex=False)
    result = result.rename(columns={aa_col: 'seq', score_col: score_col_name})
    return result

def threshold_to_label(df, score_col, threshold, label_col='label'):
    result = df.copy()
    result[label_col] = (result[score_col] >= threshold).astype(int)
    return result


oncogene_mut_df_flat=clean_flattened_df(oncogene_mut_df_flat, "escore", allele_col='allele', aa_col='AA', score_col='score')

oncogene_wt_df_flat=clean_flattened_df(oncogene_wt_df_flat, "escore", allele_col='allele', aa_col='AA', score_col='score')

# threshold based on recommended E-score value
oncogene_mut_df_flat=threshold_to_label(oncogene_mut_df_flat, "escore", 3.8, label_col='label')
oncogene_wt_df_flat=threshold_to_label(oncogene_wt_df_flat, "escore", 3.8, label_col='label')

print_data_stats(oncogene_mut_df_flat)
print_data_stats(oncogene_wt_df_flat)

Total: 75,900
Pos: 7,783 | 10.25%
Neg: 68,117 | 89.75%
N alleles: 50
Total: 75,900
Pos: 7,209 | 9.50%
Neg: 68,691 | 90.50%
N alleles: 50


In [ ]:
# merge
oncogene_mut_df_flat["mut_type"]="mut"
oncogene_wt_df_flat["mut_type"]="wt"
oncogene_mut_wt_df=pd.concat([oncogene_mut_df_flat, oncogene_wt_df_flat])
print_data_stats(oncogene_mut_wt_df)

Total: 151,800
Pos: 14,992 | 9.88%
Neg: 136,808 | 90.12%
N alleles: 50


In [ ]:
# get escape seq reported neoantigens
os.listdir(f"{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq")

escapeseq_reported_neoantigens=pd.read_excel(f"{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq/escape_seq_reported_neoantigens_fig4.xlsx")

In [ ]:
# format
escapeseq_reported_neoantigens.Allele=escapeseq_reported_neoantigens.Allele.str.replace("*", "").str.replace(":", "")
escapeseq_reported_neoantigens.rename(columns={"Peptide": "seq", "Allele": "allele", "E-score": "escore"}, inplace=True)

# add label column
escapeseq_reported_neoantigens["label"]=1

# replace allele name and reprocess
escapeseq_reported_neoantigens.allele=escapeseq_reported_neoantigens.allele.str.replace(".", "")

print_data_stats(escapeseq_reported_neoantigens)

Total: 21
Pos: 21 | 100.00%
Neg: 0 | 0.00%
N alleles: 11


In [ ]:
%%time
# expand neoantigens to all
def expand_alleles(df, seq_col='seq', allele_col='allele', list_allele_col='other_alleles'):
    def clean_allele(a):
        return a.strip().replace('*', '').replace(':', '')
    
    rows = []
    for _, row in df.iterrows():
        rows.append(row)
        
        other = row[list_allele_col]
        if pd.isna(other) or str(other).strip() == '':
            continue
        
        for allele in str(other).split(','):
            allele = clean_allele(allele)
            if not allele:
                continue
            new_row = row.copy()
            new_row[allele_col] = allele
            rows.append(new_row)
    
    return pd.DataFrame(rows).reset_index(drop=True)

escapeseq_reported_neoantigens_expanded=expand_alleles(escapeseq_reported_neoantigens, seq_col='seq', allele_col='allele', list_allele_col='Other HLA alleles (E-score >3.5)')

escapeseq_reported_neoantigens_expanded=create_seq_allele_column(escapeseq_reported_neoantigens_expanded)

escapeseq_reported_neoantigens_expanded.drop_duplicates(subset=["seq_allele"], inplace=True)

escapeseq_reported_neoantigens_expanded.reset_index(inplace=True, drop=True)

# replace allele name and reprocess
escapeseq_reported_neoantigens_expanded.allele=escapeseq_reported_neoantigens_expanded.allele.str.replace(".", "")

print_data_stats(escapeseq_reported_neoantigens_expanded)

Total: 225
Pos: 225 | 100.00%
Neg: 0 | 0.00%
N alleles: 46
CPU times: user 16.5 ms, sys: 57 µs, total: 16.6 ms
Wall time: 15.2 ms


In [ ]:
# escape seq source data
a0201_iedb_escapeseq_df = pd.read_excel(f'{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq/escape_seq_source_data.xls', sheet_name='A0201IEDB')
a0101_iedb_escapeseq_df = pd.read_excel(f'{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq/escape_seq_source_data.xls', sheet_name='A0101IEDB')
b0702_iedb_escapeseq_df = pd.read_excel(f'{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq/escape_seq_source_data.xls', sheet_name='B0702IEDB')

"""
NOTE: they use 3.2 as the threshold here as specified in the ESCAPE-seq text.
"""
# format
a0201_iedb_escapeseq_df.rename(columns={"Netscore": "el_score", "netrank": "el_rank", "AA": "seq", "score": "escore"}, inplace=True)
a0101_iedb_escapeseq_df.rename(columns={"A1el_score": "el_score", "A1el_rank": "el_rank", "A1ba_ic50": "ba_score", "A1ba_rank": "ba_rank", "AA": "seq", "score": "escore"}, inplace=True)
b0702_iedb_escapeseq_df.rename(columns={"B7el_score": "el_score", "B7el_rank": "el_rank", "B7ba_ic50": "ba_score", "B7ba_rank": "ba_rank", "AA": "seq", "score": "escore"}, inplace=True)

a0201_iedb_escapeseq_df["allele"]="A0201"
a0101_iedb_escapeseq_df["allele"]="A0101"
b0702_iedb_escapeseq_df["allele"]="B0702"

cols=["seq", "allele", "escore", "el_score", "el_rank"] #  "ba_score", "ba_rank"
iedb_escapeseq_df=pd.concat([a0201_iedb_escapeseq_df[cols], a0101_iedb_escapeseq_df[cols], b0702_iedb_escapeseq_df[cols]])

iedb_escapeseq_df_processed=full_process_data(iedb_escapeseq_df, mhc_seq_dict, remove_flank=False, add_pseudo_label=False)
iedb_escapeseq_df_processed["peptide"]=iedb_escapeseq_df_processed.seq.str.replace("$", "")
iedb_escapeseq_df_processed=create_seq_allele_column(iedb_escapeseq_df_processed)
iedb_escapeseq_df_processed=threshold_to_label(iedb_escapeseq_df_processed, "escore", 3.2, label_col='label')
print_data_stats(iedb_escapeseq_df_processed)

Total: 4,479
Pos: 1,445 | 32.26%
Neg: 3,034 | 67.74%
N alleles: 3


In [ ]:
%%time
# get covid set
# escape seq source data
a0201_sars_escapeseq_df = pd.read_excel(f'{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq/escape_seq_source_data.xls', sheet_name='A0201SAR2')
a0101_sars_escapeseq_df = pd.read_excel(f'{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq/escape_seq_source_data.xls', sheet_name='A0101SAR2')
b0702_sars_escapeseq_df = pd.read_excel(f'{DATA_DIR}/Training_Datasets/Collected/ESCAPE_seq/escape_seq_source_data.xls', sheet_name='B0702ASAR2')

# format
a0201_sars_escapeseq_df.rename(columns={"A2el_score": "el_score", "A2el_rank": "el_rank", "A2ba_ic50": "ba_score", "A2ba_rank": "ba_rank", "AA": "seq", "score": "escore"}, inplace=True)
a0101_sars_escapeseq_df.rename(columns={"A1el_score": "el_score", "A1el_rank": "el_rank", "A1ba_ic50": "ba_score", "A1ba_rank": "ba_rank", "AA": "seq", "score": "escore"}, inplace=True)
b0702_sars_escapeseq_df.rename(columns={"B7el_score": "el_score", "B7el_rank": "el_rank", "B7ba_ic50": "ba_score", "B7ba_rank": "ba_rank", "AA": "seq", "score": "escore"}, inplace=True)

a0201_sars_escapeseq_df["allele"]="A0201"
a0101_sars_escapeseq_df["allele"]="A0101"
b0702_sars_escapeseq_df["allele"]="B0702"

cols=["seq", "allele", "escore", "el_score", "el_rank", "ba_score", "ba_rank"]
sars_escapeseq_df=pd.concat([a0201_sars_escapeseq_df[cols], a0101_sars_escapeseq_df[cols], b0702_sars_escapeseq_df[cols]])

sars_escapeseq_df_processed=full_process_data(sars_escapeseq_df, mhc_seq_dict, remove_flank=False, add_pseudo_label=False)
sars_escapeseq_df_processed["peptide"]=sars_escapeseq_df_processed.seq.str.replace("$", "")
sars_escapeseq_df_processed=create_seq_allele_column(sars_escapeseq_df_processed)
sars_escapeseq_df_processed=threshold_to_label(sars_escapeseq_df_processed, "escore", 3.2, label_col='label')
print_data_stats(sars_escapeseq_df_processed)

Total: 8,001
Pos: 263 | 3.29%
Neg: 7,738 | 96.71%
N alleles: 3
CPU times: user 1.96 s, sys: 59.6 ms, total: 2.02 s
Wall time: 2.01 s


In [ ]:
%%time
"""
Save ESCAPE-seq datasets
"""
escape_seq_datasets_dir=f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq"
os.makedirs(escape_seq_datasets_dir, exist_ok=True)

# oncogene_mut_wt_df.to_csv(f"{escape_seq_datasets_dir}/oncogene_mut_wt_df.csv")

# oncogene_mut_df_flat.to_csv(f"{escape_seq_datasets_dir}/oncogene_mut_df_flat.csv")
# oncogene_wt_df_flat.to_csv(f"{escape_seq_datasets_dir}/oncogene_wt_df_flat.csv")

# escapeseq_reported_neoantigens.to_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens.csv")
# escapeseq_reported_neoantigens_expanded.to_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded.csv")

# cov/iedb sets
# iedb_escapeseq_df_processed.to_csv(f"{escape_seq_datasets_dir}/iedb_escapeseq_df_processed.csv")
# sars_escapeseq_df_processed.to_csv(f"{escape_seq_datasets_dir}/sars_escapeseq_df_processed.csv")

CPU times: user 39.7 ms, sys: 14.5 ms, total: 54.2 ms
Wall time: 53.1 ms


### Merge MS data and process with respect to non-MS data

 - Generate MS train and test sets.

In [ ]:
ms_datasets={
        "iedb": iedb_f_ms_pos, 
        "mhcma": mhcma,
        "hlathena_data": hlathena_data,
        "nmp": nmp_el_data_hla,
        "mhcf_s2": s2,
        "mhcf_s3_ms": s3_ms,
        "bm1_train": bm1_train_sa,
        "bm1_test": bm1_test_sa,
}

nonms_datasets={
        "yd": holec_et_al_pathogen_data, # yeast display data
        "episcan_r9": sd2_formatted_spp_ko, 
        "episcan_rt": sd6_df, 
        "episcan_cov": sd3_formatted, 
        "escape_seq_paired_mut": oncogene_mut_wt_df,
        "escape_seq_reported_neoantigens": escapeseq_reported_neoantigens_expanded,
        "ba": nmp_ba_data_hla, # binding affinity data
}

In [ ]:
%%time
# get ms eluted ligand binders
positive_seqs_ms=pd.concat([iedb_f_ms_pos, mhcma.query("label==1"), hlathena_data.query("label==1"), nmp_el_data_hla.query("label==1"), s2.query("label==1"), s3_ms.query("label==1"), bm1_train_sa.query("label==1"), bm1_test_sa.query("label==1")])
positive_seqs_ms=create_seq_allele_column(positive_seqs_ms)
positive_seqs_ms.method_group="MS"
positive_seqs_ms=filter_seq_len(positive_seqs_ms)
positive_seqs_ms.drop_duplicates(subset=["seq_allele"], inplace=True)

# filter lengths
positive_seqs_ms["seq_len"]=positive_seqs_ms.seq.apply(len)
positive_seqs_ms=positive_seqs_ms.query("seq_len >= 8 and seq_len <= 14")

# print stats
print_data_stats(positive_seqs_ms)
print(positive_seqs_ms.allele.nunique())

Total: 778,171
Pos: 778,171 | 100.00%
Neg: 0 | 0.00%
N alleles: 192
192
CPU times: user 4.18 s, sys: 1.09 s, total: 5.27 s
Wall time: 5.27 s


In [ ]:
%%time
"""
Save all ms positives (unprocessed, will need to process later).
"""
ms_train_test_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets"
os.makedirs(ms_train_test_dir, exist_ok=True)

# positive_seqs_ms.to_csv(f"{ms_train_test_dir}/positive_seqs_ms.csv")

CPU times: user 7.86 s, sys: 200 ms, total: 8.06 s
Wall time: 8.06 s


In [ ]:
%%time
"""
Remove overlaps with external sets

- Remove overlaps with YD, Epi R9, Epi Cov, ESCAPE-seq paired mut, BA, and MS pathogen sets. The MS pathogen sets and ESCAPE-seq are cleaned at a raw sequence level since they involve a ranking and recovery tasks. 
"""

# remove seq-allele overlap between MS and yd, episcan r9, episcan cov, amd ba from ms data
positive_seqs_ms=create_seq_allele_column(positive_seqs_ms)
holec_et_al_pathogen_data=create_seq_allele_column(holec_et_al_pathogen_data)
sd2_formatted_spp_ko=create_seq_allele_column(sd2_formatted_spp_ko)
sd3_formatted=create_seq_allele_column(sd3_formatted)
nmp_ba_data_hla=create_seq_allele_column(nmp_ba_data_hla)

seq_allele_overlap=df_col_set_intersection(positive_seqs_ms, holec_et_al_pathogen_data, "seq_allele", print_output=True, return_output=True)
positive_seqs_ms=positive_seqs_ms[~positive_seqs_ms.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(positive_seqs_ms, sd2_formatted_spp_ko, "seq_allele", print_output=True, return_output=True)
positive_seqs_ms=positive_seqs_ms[~positive_seqs_ms.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(positive_seqs_ms, sd3_formatted, "seq_allele", print_output=True, return_output=True)
positive_seqs_ms=positive_seqs_ms[~positive_seqs_ms.seq_allele.isin(seq_allele_overlap)]

seq_allele_overlap=df_col_set_intersection(positive_seqs_ms, nmp_ba_data_hla, "seq_allele", print_output=True, return_output=True)
positive_seqs_ms=positive_seqs_ms[~positive_seqs_ms.seq_allele.isin(seq_allele_overlap)]

print_data_stats(positive_seqs_ms)
print(positive_seqs_ms.allele.nunique())

39
0
13
1056
Total: 777,063
Pos: 777,063 | 100.00%
Neg: 0 | 0.00%
N alleles: 192
192
CPU times: user 3.68 s, sys: 729 ms, total: 4.41 s
Wall time: 4.41 s


In [ ]:
# remove raw sequence overlap between ms and escape-seq, pathogen ms sets, and aav ms set
seq_overlap=df_col_set_intersection(positive_seqs_ms, oncogene_mut_wt_df, "seq", print_output=True, return_output=True)
positive_seqs_ms=positive_seqs_ms[~positive_seqs_ms.seq.isin(seq_overlap)]

seq_overlap=df_col_set_intersection(positive_seqs_ms, ms_cov_set, "seq", print_output=True, return_output=True)
positive_seqs_ms=positive_seqs_ms[~positive_seqs_ms.seq.isin(seq_overlap)]

seq_overlap=df_col_set_intersection(positive_seqs_ms, ms_zika_set, "seq", print_output=True, return_output=True)
positive_seqs_ms=positive_seqs_ms[~positive_seqs_ms.seq.isin(seq_overlap)]

seq_overlap=df_col_set_intersection(positive_seqs_ms, ms_mtb_set, "seq", print_output=True, return_output=True)
positive_seqs_ms=positive_seqs_ms[~positive_seqs_ms.seq.isin(seq_overlap)]

print_data_stats(positive_seqs_ms)
print(positive_seqs_ms.allele.nunique())

27
0
0
0
Total: 777,016
Pos: 777,016 | 100.00%
Neg: 0 | 0.00%
N alleles: 192
192


In [ ]:
# check reported neoantigens
seq_overlap=df_col_set_intersection(positive_seqs_ms, escapeseq_reported_neoantigens_expanded, "seq", print_output=True, return_output=True)

0


In [ ]:
# remove episcan rt seq allele overlaps from episcan rt (we can also do this later)
seq_allele_overlap=df_col_set_intersection(positive_seqs_ms, sd6_df, "seq_allele", print_output=True, return_output=True)
sd6_df_cleaned=sd6_df[~sd6_df.seq_allele.isin(seq_allele_overlap)]

36562


In [ ]:
%%time
# generate synthetic negatives from human proteome
chp_filtered_negative_peptides=generate_kmers_from_proteome(canonical_human_proteins, kmer_lengths=[8,9,10,11,12,13,14])
random_negative_peptides=generate_random_peptides(n_peptides=10, lengths=[8,9,10,11,12,13,14])

chp_filtered_negative_peptides_no_dup={}
for l,p in tqdm(chp_filtered_negative_peptides.items()):
    chp_filtered_negative_peptides_no_dup[l]=list(set(p))
    print("Length", l)
    print(len(chp_filtered_negative_peptides[l]))
    print(len(chp_filtered_negative_peptides_no_dup[l]))

 29%|██▊       | 2/7 [00:20<00:50, 10.19s/it]

Length 8
11250173
10300888
Length 9
11229814
10405224


 57%|█████▋    | 4/7 [00:40<00:30, 10.02s/it]

Length 10
11209455
10466109
Length 11
11189096
10509168


 71%|███████▏  | 5/7 [00:50<00:20, 10.12s/it]

Length 12
11168738
10540392


100%|██████████| 7/7 [01:11<00:00, 10.15s/it]

Length 13
11148381
10563518
Length 14
11128026
10580237


100%|██████████| 7/7 [01:11<00:00, 10.14s/it]

CPU times: user 1min 56s, sys: 6.53 s, total: 2min 2s
Wall time: 2min 2s


In [ ]:
%%time
"""
Remove sequences from dict that we don't want overlap with at all.
"""
# exclude peptides from negatives
exclude_seqs=oncogene_mut_wt_df.seq.tolist() + ms_cov_set.seq.tolist() + ms_zika_set.seq.tolist() + ms_mtb_set.seq.tolist()

chp_filtered_negative_peptides_no_dup_cleaned=filter_dict_by_peptides(chp_filtered_negative_peptides_no_dup, exclude_seqs)

CPU times: user 15.2 s, sys: 2.3 s, total: 17.5 s
Wall time: 17.5 s


In [ ]:
%%time
"""
Generate MS train and test sets, excluding seq_allele from any other dataset when sampling synthetic negatives
"""

holec_et_al_pathogen_data.reset_index(inplace=True, drop=True)
sd2_formatted_spp_ko.reset_index(inplace=True, drop=True)
sd6_df_cleaned.reset_index(inplace=True, drop=True)
sd3_formatted.reset_index(inplace=True, drop=True)
oncogene_mut_wt_df.reset_index(inplace=True, drop=True)
escapeseq_reported_neoantigens_expanded.reset_index(inplace=True, drop=True)
nmp_ba_data_hla.reset_index(inplace=True, drop=True)

seq_allele_datasets_to_exclude={
        "yd": holec_et_al_pathogen_data, # yeast display data
        "episcan_r9": sd2_formatted_spp_ko, 
        # "episcan_rt": sd6_df, 
        "episcan_rt": sd6_df_cleaned, # use cleaned version
        "episcan_cov": sd3_formatted, 
        "escape_seq_paired_mut": oncogene_mut_wt_df,
        "escape_seq_reported_neoantigens": escapeseq_reported_neoantigens_expanded,
        "ba": nmp_ba_data_hla, # binding affinity data
}

ms_train_data, ms_test_data = split_ms_with_synthetic_negatives(
    ms_pos_df=positive_seqs_ms,
    other_dfs=seq_allele_datasets_to_exclude,
    synthetic_negatives_by_length=chp_filtered_negative_peptides_no_dup_cleaned,
    test_frac=0.10,
    min_test_per_group=5,
    max_test_per_group=100, # use smaller max
    train_neg_per_pos=50,
    test_neg_per_pos=99,
    seed=42,
)

Sampling synthetic negatives: 100%|██████████| 1084/1084 [00:33<00:00, 32.28it/s] 


CPU times: user 58.2 s, sys: 7.38 s, total: 1min 5s
Wall time: 1min 5s


In [ ]:
# print stats
print_data_stats(ms_train_data)
print(ms_train_data.allele.nunique())
print(end="\n")
print_data_stats(ms_test_data)
print(ms_test_data.allele.nunique())

Total: 37,988,523
Pos: 744,873 | 1.96%
Neg: 37,243,650 | 98.04%
N alleles: 192
192

Total: 3,214,300
Pos: 32,143 | 1.00%
Neg: 3,182,157 | 99.00%
N alleles: 142
142


In [ ]:
%%time
"""
Final check to ensure no seq-allele or seq overlaps
"""
ms_train_data=create_seq_allele_column(ms_train_data)
ms_test_data=create_seq_allele_column(ms_test_data)

seq_allele_overlap=df_col_set_intersection(ms_train_data, holec_et_al_pathogen_data, "seq_allele", print_output=True, return_output=True)

seq_allele_overlap=df_col_set_intersection(ms_train_data, sd2_formatted_spp_ko, "seq_allele", print_output=True, return_output=True)

seq_allele_overlap=df_col_set_intersection(ms_train_data, sd3_formatted, "seq_allele", print_output=True, return_output=True)

seq_allele_overlap=df_col_set_intersection(ms_train_data, nmp_ba_data_hla, "seq_allele", print_output=True, return_output=True)

seq_allele_overlap=df_col_set_intersection(ms_train_data, sd6_df_cleaned, "seq_allele", print_output=True, return_output=True) # use cleaned version

seq_overlap=df_col_set_intersection(ms_train_data, ms_test_data, "seq_allele", print_output=True, return_output=True)

seq_overlap=df_col_set_intersection(ms_train_data, oncogene_mut_wt_df, "seq", print_output=True, return_output=True)

seq_overlap=df_col_set_intersection(ms_train_data, escapeseq_reported_neoantigens_expanded, "seq", print_output=True, return_output=True)

seq_overlap=df_col_set_intersection(ms_train_data, ms_cov_set, "seq", print_output=True, return_output=True)

seq_overlap=df_col_set_intersection(ms_train_data, ms_zika_set, "seq", print_output=True, return_output=True)

seq_overlap=df_col_set_intersection(ms_train_data, ms_mtb_set, "seq", print_output=True, return_output=True)

0
0
0
0
0
0
0
0
0
0
0
0
CPU times: user 2min 57s, sys: 18.5 s, total: 3min 15s
Wall time: 3min 15s


In [ ]:
%%time
"""
Save MS train/test sets
"""
ms_train_test_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets"
os.makedirs(ms_train_test_dir, exist_ok=True)

# ms_train_data.to_csv(f"{ms_train_test_dir}/ms_train_data.csv")
# ms_test_data.to_csv(f"{ms_train_test_dir}/ms_test_data.csv")

CPU times: user 15 µs, sys: 2 µs, total: 17 µs
Wall time: 32.7 µs


### Yeast display train and test sets

 - Remove the pathogen MS sets at the sequence level.

In [ ]:
# read holec data and ms pathogen sets

holec_et_al_pathogen_data=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/Holec_et_al_2025/holec_et_al_pathogen_data.csv", index_col=0)

ms_cov_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Pathogen_Sets/ms_cov_set.csv", index_col=0)
ms_zika_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Pathogen_Sets/ms_zika_set.csv", index_col=0)
ms_mtb_set=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Pathogen_Sets/ms_mtb_set.csv", index_col=0)

In [ ]:
# clean seq overlap
seq_overlap=df_col_set_intersection(ms_cov_set, holec_et_al_pathogen_data, "seq", print_output=True, return_output=True)
holec_et_al_pathogen_data_cleaned=holec_et_al_pathogen_data[~holec_et_al_pathogen_data.seq.isin(seq_overlap)]

seq_overlap=df_col_set_intersection(ms_zika_set, holec_et_al_pathogen_data_cleaned, "seq", print_output=True, return_output=True)
holec_et_al_pathogen_data_cleaned=holec_et_al_pathogen_data_cleaned[~holec_et_al_pathogen_data_cleaned.seq.isin(seq_overlap)]

seq_overlap=df_col_set_intersection(ms_mtb_set, holec_et_al_pathogen_data_cleaned, "seq", print_output=True, return_output=True)
holec_et_al_pathogen_data_cleaned=holec_et_al_pathogen_data_cleaned[~holec_et_al_pathogen_data_cleaned.seq.isin(seq_overlap)]

6
7
6


In [ ]:
# split train and test for holec data
# take 100 peptides from each alleles as positives and then x99 for negatives
n_pos=100
pos_neg_ratio=99
holec_train_set, holec_test_set=train_test_split_by_allele(holec_et_al_pathogen_data_cleaned, "seq", "allele", "label", n_pos, pos_neg_ratio, seed=42)

print_data_stats(holec_train_set)
print_data_stats(holec_test_set)

Total: 1,392,048
Pos: 26,883 | 1.93%
Neg: 1,365,165 | 98.07%
N alleles: 16
Total: 160,000
Pos: 1,600 | 1.00%
Neg: 158,400 | 99.00%
N alleles: 16


In [ ]:
"""
Save train and test sets
"""
# holec_train_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/Holec_et_al_2025/holec_train_set.csv")
# holec_test_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/Holec_et_al_2025/holec_test_set.csv")

### EpiScan train and test sets

In [ ]:
"""
Create EpiScan train and test sets
"""

# first, remove overlaps between the cov set and episcan r9 and rt sets from the r9 and rt sets
seq_allele_overlap=df_col_set_intersection(sd3_formatted, sd2_formatted_spp_ko, "seq_allele", print_output=True, return_output=True)

seq_allele_overlap=df_col_set_intersection(sd3_formatted, sd6_df_cleaned, "seq_allele", print_output=True, return_output=True)
sd6_df_cleaned_v2=sd6_df_cleaned[~sd6_df_cleaned.seq_allele.isin(seq_allele_overlap)]

0
350


In [ ]:
# split train and test for EpiScan data
# take 100 peptides from each alleles as positives and then x99 for negatives
n_pos=100
pos_neg_ratio=99
episcan_r9_train_set, episcan_r9_test_set=train_test_split_by_allele(sd2_formatted_spp_ko, "seq", "allele", "label", n_pos, pos_neg_ratio, seed=42)

episcan_rt_train_set, episcan_rt_test_set=train_test_split_by_allele(sd6_df_cleaned_v2, "seq", "allele", "label", n_pos, pos_neg_ratio, seed=42)

print(end="\n")

print_data_stats(episcan_r9_train_set)
print_data_stats(episcan_r9_test_set)

print(end="\n")

print_data_stats(episcan_rt_train_set)
print_data_stats(episcan_rt_test_set)


Total: 1,879,616
Pos: 3,818 | 0.20%
Neg: 1,875,798 | 99.80%
N alleles: 4
Total: 40,000
Pos: 400 | 1.00%
Neg: 39,600 | 99.00%
N alleles: 4

Total: 652,761
Pos: 104,678 | 16.04%
Neg: 548,083 | 83.96%
N alleles: 4
Total: 40,000
Pos: 400 | 1.00%
Neg: 39,600 | 99.00%
N alleles: 4


In [ ]:
# ensure no cross train/test set overlap
seq_allele_overlap=df_col_set_intersection(episcan_r9_train_set, episcan_rt_test_set, "seq_allele", print_output=True, return_output=True)

# remove
episcan_r9_train_set=episcan_r9_train_set[~episcan_r9_train_set.seq_allele.isin(seq_allele_overlap)]

print_data_stats(episcan_r9_train_set)

seq_allele_overlap=df_col_set_intersection(episcan_rt_train_set, episcan_r9_test_set, "seq_allele", print_output=True, return_output=True)

1
Total: 1,879,615
Pos: 3,818 | 0.20%
Neg: 1,875,797 | 99.80%
N alleles: 4
0


In [ ]:
# reset indices
episcan_r9_train_set.reset_index(inplace=True, drop=True)
episcan_r9_test_set.reset_index(inplace=True, drop=True)
episcan_rt_train_set.reset_index(inplace=True, drop=True)
episcan_rt_test_set.reset_index(inplace=True, drop=True)
sd3_formatted.reset_index(inplace=True, drop=True)

In [ ]:
%%time
"""
Save EpiScan datasets
"""
episcan_train_test_dir=f"{DATA_DIR}/Training_Datasets/Processed/EpiScan"
os.makedirs(episcan_train_test_dir, exist_ok=True)

# episcan_r9_train_set.to_csv(f"{episcan_train_test_dir}/episcan_r9_train_set.csv")
# episcan_r9_test_set.to_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv")

# episcan_rt_train_set.to_csv(f"{episcan_train_test_dir}/episcan_rt_train_set.csv")
# episcan_rt_test_set.to_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv")

# sd3_formatted.to_csv(f"{episcan_train_test_dir}/sd3_formatted.csv")

CPU times: user 7 µs, sys: 1e+03 ns, total: 8 µs
Wall time: 15 µs


'\nSave EpiScan datasets\n\nNOTE: need to be processed.\n'

In [ ]:
"""
NOTE:
# CREATE TEST SETS FOR EPISCAN THAT HAVE MORE CYSTEINES AND MORE HYDROPHOBIC PEPTIDES, ALONG WITH VL TRANSITION OR PROLINES.
# 
"""

'\nNOTE:\n# CREATE TEST SETS FOR EPISCAN THAT HAVE MORE CYSTEINES AND MORE HYDROPHOBIC PEPTIDES, ALONG WITH VL TRANSITION OR PROLINES.\n# \n'

### BA train and test sets

 - For updating, we will use a much smaller fraction of data, and split at 1:1 for the appropriate testing of Spearman correlation. Dataset is too small to support 1:99 test and an additional train set.
 - For general benchmarking of MHCPrime and other models, we will use a much larger set naturally.
 - Split for MHCPrime benchmarking at a 1:99 positive to negative ratio (matched to MS allele-length), a set for unseen distributions, and a set for testing ranking (1:1 positive to negative ratio).

In [ ]:
# import processed ba data and ms test set for allele matching
nmp_ba_data_hla=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/nmp_ba_data_hla.csv", index_col=0)
ms_test_data=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/ms_test_data.csv", index_col=0)

In [ ]:
# get seq len
nmp_ba_data_hla["seq_len"]=nmp_ba_data_hla.seq.str.replace("$", "").apply(len)
ms_test_data["seq_len"]=ms_test_data.seq.str.replace("$", "").apply(len)

In [ ]:
# get allele length matched ba data
nmp_ba_data_hla_allele_length_matched=filter_to_shared_pairs(nmp_ba_data_hla, ms_test_data, allele_col='allele', seq_len_col='seq_len')

In [ ]:
# sample ba data for the base mhcprime benchmarking task

# classification dataset at a 1:99 pos to neg ratio
ba_full_cls_test_set, _=sample_max_ratio(nmp_ba_data_hla_allele_length_matched, neg_ratio=99, label_col='label', allele_col='allele', seq_col='seq', seed=42)
print_data_stats(ba_full_cls_test_set)

# continuous dataset at a 1:1 pos to neg ratio
ba_full_con_test_set, _=sample_max_ratio(nmp_ba_data_hla_allele_length_matched, neg_ratio=1, label_col='label', allele_col='allele', seq_col='seq', seed=42)
print_data_stats(ba_full_con_test_set)

Sampling alleles: 100%|██████████| 83/83 [00:00<00:00, 389.23it/s]


Total: 111,500
Pos: 1,115 | 1.00%
Neg: 110,385 | 99.00%
N alleles: 71


Sampling alleles: 100%|██████████| 83/83 [00:00<00:00, 430.39it/s]


Total: 79,578
Pos: 39,789 | 50.00%
Neg: 39,789 | 50.00%
N alleles: 76


In [ ]:
"""
Save sets
"""
# ba_full_cls_test_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_cls_test_set.csv")
# ba_full_con_test_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/ba_full_con_test_set.csv")

'\nSave sets\n'

In [ ]:
%%time
# split ba data for updating
pct=0.05
min_count=40
max_count=200

train_df_ba_for_updating, test_df_ba_for_updating=split_percentage_with_bounds(nmp_ba_data_hla_allele_length_matched, pct, min_count, max_count, label_col='label', allele_col='allele', seq_col='seq', neg_ratio=1)

print_data_stats(train_df_ba_for_updating)
print("Train N alleles: ", train_df_ba.allele.nunique())
print_data_stats(test_df_ba_for_updating)
print("Test N alleles: ", test_df_ba.allele.nunique())

Splitting alleles: 100%|██████████| 83/83 [00:00<00:00, 335.26it/s]

Total: 151,064
Pos: 37,770 | 25.00%
Neg: 113,294 | 75.00%
N alleles: 83
Train N alleles:  83
Total: 4,221
Pos: 2,077 | 49.21%
Neg: 2,144 | 50.79%
N alleles: 51
Test N alleles:  51
CPU times: user 308 ms, sys: 7.76 ms, total: 316 ms
Wall time: 313 ms


In [ ]:
"""
Save sets
"""
# train_df_ba_for_updating.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/train_df_ba_for_updating.csv")
# test_df_ba_for_updating.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/BA/test_df_ba_for_updating.csv")

'\nSave sets\n'

### ESCAPE-seq train and test sets

 - For general benchmarking of MHCPrime and other models, we will use all the data.
 - Same as above, split at 1:99 and another set at 1:1, with matched allele-length as MS.

In [ ]:
# import processed escape-seq data and ms test set for allele matching
oncogene_mut_wt_df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/oncogene_mut_wt_df.csv", index_col=0)
ms_test_data=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/ms_test_data.csv", index_col=0)

In [ ]:
# process
oncogene_mut_wt_df=full_process_data(oncogene_mut_wt_df, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
oncogene_mut_wt_df=create_seq_allele_column(oncogene_mut_wt_df)

ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

In [ ]:
# get seq len
oncogene_mut_wt_df["seq_len"]=oncogene_mut_wt_df.seq.str.replace("$", "").apply(len)
ms_test_data["seq_len"]=ms_test_data.seq.str.replace("$", "").apply(len)

In [ ]:
# get allele length matched escape-seq data
oncogene_mut_wt_df_allele_length_matched=filter_to_shared_pairs(oncogene_mut_wt_df, ms_test_data, allele_col='allele', seq_len_col='seq_len')
oncogene_mut_wt_df_allele_length_matched.drop_duplicates(subset=["seq_allele"], inplace=True)

In [ ]:
# filter x and a68
oncogene_mut_wt_df_allele_length_matched_noX=filter_sequences_without_x(oncogene_mut_wt_df_allele_length_matched)
oncogene_mut_wt_df_allele_length_matched_noX_noA68=oncogene_mut_wt_df_allele_length_matched_noX.query("allele != 'A6801'")

In [ ]:
# sample ba data for the base mhcprime benchmarking task

# classification dataset at a 1:99 pos to neg ratio
escape_seq_full_cls_test_set, _=sample_max_ratio(oncogene_mut_wt_df_allele_length_matched_noX_noA68, neg_ratio=99, label_col='label', allele_col='allele', seq_col='seq', seed=42)
print_data_stats(escape_seq_full_cls_test_set)

# continuous dataset at a 1:1 pos to neg ratio
escape_seq_full_con_test_set, _=sample_max_ratio(oncogene_mut_wt_df_allele_length_matched_noX_noA68, neg_ratio=1, label_col='label', allele_col='allele', seq_col='seq', seed=42)
print_data_stats(escape_seq_full_con_test_set)

Sampling alleles: 100%|██████████| 44/44 [00:00<00:00, 268.44it/s]


Total: 55,900
Pos: 559 | 1.00%
Neg: 55,341 | 99.00%
N alleles: 44


Sampling alleles: 100%|██████████| 44/44 [00:00<00:00, 316.04it/s]


Total: 14,456
Pos: 7,228 | 50.00%
Neg: 7,228 | 50.00%
N alleles: 44


In [ ]:
"""
Save sets
"""
# escape_seq_full_cls_test_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_seq_full_cls_test_set.csv")
# escape_seq_full_con_test_set.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq/escape_seq_full_con_test_set.csv")

### Split Ligand.MHC Atlas data

In [ ]:
# ms train and test sets
datasets_dir = f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

In [ ]:
# escape seq sets
escape_seq_datasets_dir = f"{DATA_DIR}/Training_Datasets/Processed/ESCAPE_seq"
oncogene_mut_wt_df=pd.read_csv(f"{escape_seq_datasets_dir}/oncogene_mut_wt_df.csv", index_col=0)
escapeseq_reported_neoantigens=pd.read_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens.csv", index_col=0)
escapeseq_reported_neoantigens_expanded=pd.read_csv(f"{escape_seq_datasets_dir}/escapeseq_reported_neoantigens_expanded.csv", index_col=0)

In [ ]:
# create version without reported neoantigens
oncogene_mut_wt_df.seq=oncogene_mut_wt_df.seq.str.replace("$", "")
escapeseq_reported_neoantigens_expanded.seq=escapeseq_reported_neoantigens_expanded.seq.str.replace("$", "")
oncogene_mut_wt_df=create_seq_allele_column(oncogene_mut_wt_df)
escapeseq_reported_neoantigens_expanded=create_seq_allele_column(escapeseq_reported_neoantigens_expanded)
seq_overlap=df_col_set_intersection(oncogene_mut_wt_df, escapeseq_reported_neoantigens_expanded, "seq", print_output=True, return_output=True)
oncogene_mut_wt_df_cleaned=oncogene_mut_wt_df[~oncogene_mut_wt_df.seq.isin(seq_overlap)]
oncogene_mut_wt_df_cleaned_dedup=oncogene_mut_wt_df_cleaned.drop_duplicates(subset=["seq_allele"])

20


In [ ]:
# merge to create train set
oncogene_mut_wt_df_cleaned_dedup.rename(columns={"escore": "ba_value"}, inplace=True)
oncogene_mut_wt_df_cleaned_dedup["domain_id"]=1
ms_train_data["domain_id"]=0
ms_escape_seq_train_data=pd.concat([ms_train_data, oncogene_mut_wt_df_cleaned_dedup])
ms_escape_seq_train_data.reset_index(inplace=True, drop=True)
print_data_stats(ms_escape_seq_train_data)

Total: 38,062,973
Pos: 752,461 | 1.98%
Neg: 37,310,512 | 98.02%
N alleles: 197


In [ ]:
# remove X containing seqs and A68 sequences
def filter_sequences_without_x(df, seq_col='seq'):
    return df[~df[seq_col].str.contains('X', na=False)].reset_index(drop=True)

ms_escape_seq_train_data_noX=filter_sequences_without_x(ms_escape_seq_train_data)
ms_escape_seq_train_data_noX_noA68=ms_escape_seq_train_data_noX.query("allele != 'A6801'")
ms_escape_seq_train_data_noX_noA68.reset_index(inplace=True, drop=True)
print_data_stats(ms_escape_seq_train_data_noX_noA68)

Total: 37,404,215
Pos: 739,533 | 1.98%
Neg: 36,664,682 | 98.02%
N alleles: 196


In [ ]:
# match alleles that are present in ms (for new alleles, its fine to keep them in)
ms_escape_seq_train_data_noX_noA68["seq_len"]=ms_escape_seq_train_data_noX_noA68.seq.str.replace("$", "").apply(len)
ms_escape_seq_train_data_noX_noA68_matched_allele=ms_escape_seq_train_data_noX_noA68[ms_escape_seq_train_data_noX_noA68.allele.isin(oncogene_mut_wt_df_cleaned_dedup.allele.unique().tolist())].query("seq_len == 9")
ms_escape_seq_train_data_noX_noA68_matched_allele.reset_index(inplace=True, drop=True)
print_data_stats(ms_escape_seq_train_data_noX_noA68_matched_allele)

Total: 12,340,124
Pos: 248,092 | 2.01%
Neg: 12,092,032 | 97.99%
N alleles: 49


In [ ]:
# full process
ms_escape_seq_train_data_noX_noA68_matched_allele=full_process_data(ms_escape_seq_train_data_noX_noA68_matched_allele, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)

escapeseq_reported_neoantigens=full_process_data(escapeseq_reported_neoantigens, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)

escapeseq_reported_neoantigens_expanded=full_process_data(escapeseq_reported_neoantigens_expanded, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)

In [ ]:
# ligand atlas data
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
ligand_mhc_atlas_atlas_cancer_df_processed=pd.read_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_atlas_cancer_df_processed.csv", index_col=0)

In [ ]:
%%time
# parse sample col and get cancer samples only
def parse_sample_col(df, col, new_cols):
    vals = df[col].values
    split1 = np.char.split(vals.astype(str), '::')
    part0 = np.array([s[0] for s in split1])
    part1 = np.array(['::'.join(s[1:]) for s in split1])
    split2 = np.char.split(part1, ':')
    df[new_cols[0]] = part0
    df[new_cols[1]] = np.array([s[0] for s in split2])
    df[new_cols[2]] = np.array([':'.join(s[1:]) for s in split2])
    return df

ligand_mhc_atlas_atlas_cancer_df_processed_parsed=parse_sample_col(ligand_mhc_atlas_atlas_cancer_df_processed, "Sample", ["cancer_normal", "cancer_type", "remaining_sample_info"])
ligand_mhc_atlas_atlas_cancer_df_processed_cancer=ligand_mhc_atlas_atlas_cancer_df_processed.query("cancer_normal != 'Normal'")

CPU times: user 30.5 s, sys: 6.83 s, total: 37.3 s
Wall time: 37.3 s


In [ ]:
# procses
ligand_mhc_atlas_atlas_cancer_df_processed_cancer=full_process_data(ligand_mhc_atlas_atlas_cancer_df_processed_cancer, mhc_seq_dict, remove_flank=False, add_pseudo_label=False)
ligand_mhc_atlas_atlas_cancer_df_processed_cancer=create_seq_allele_column(ligand_mhc_atlas_atlas_cancer_df_processed_cancer)
ligand_mhc_atlas_atlas_cancer_df_processed_cancer.drop_duplicates(subset=["seq_allele"], inplace=True)
print_data_stats(ligand_mhc_atlas_atlas_cancer_df_processed_cancer)

# clean
seq_overlap=df_col_set_intersection(ms_escape_seq_train_data_noX_noA68_matched_allele, ligand_mhc_atlas_atlas_cancer_df_processed_cancer, "seq", print_output=True, return_output=True)

ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned=ligand_mhc_atlas_atlas_cancer_df_processed_cancer[~ligand_mhc_atlas_atlas_cancer_df_processed_cancer.seq.isin(seq_overlap)]

print_data_stats(ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned)

# match allele and length dist
ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned["seq_len"]=ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned.seq.str.replace("$", "").apply(len)

ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned_matched_allele_length=ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned[ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned.allele.isin(ms_escape_seq_train_data_noX_noA68_matched_allele.allele.unique().tolist())].query("seq_len == 9")

ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned_matched_allele_length.reset_index(inplace=True, drop=True)

print_data_stats(ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned_matched_allele_length)

Total: 1,083,219
Pos: 1,083,219 | 100.00%
Neg: 0 | 0.00%
N alleles: 129
144610
Total: 729,807
Pos: 729,807 | 100.00%
Neg: 0 | 0.00%
N alleles: 129
Total: 29,674
Pos: 29,674 | 100.00%
Neg: 0 | 0.00%
N alleles: 41


In [ ]:
"""
Save
"""
ligand_mhc_atlas_output_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Ligand_MHC_Atlas"
# ligand_mhc_atlas_atlas_cancer_df_processed_cancer_cleaned_matched_allele_length.to_csv(f"{ligand_mhc_atlas_output_dir}/ligand_mhc_atlas_cancer_test_set_for_escape_seq.csv")

### MS 10% split in-distribution train and test sets

In [ ]:
%%time
# read in ms train and test sets
ms_train_test_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{ms_train_test_dir}/ms_train_data.csv", index_col=0)
ms_train_data.reset_index(inplace=True, drop=True)

CPU times: user 24.7 s, sys: 5.45 s, total: 30.1 s
Wall time: 30.1 s


In [ ]:
# split off percentage of data for ft
ft_percentage=0.90

ms_train_data_pt, ms_train_data_ft=split_df_by_percentage(ms_train_data, ft_percentage, min_positive_count=50, allele_col='allele', label_col='label', seed=42)

print_data_stats(ms_train_data_pt)
print_data_stats(ms_train_data_ft)

100%|██████████| 192/192 [00:00<00:00, 1055.64it/s]


Total: 3,861,640
Pos: 75,769 | 1.96%
Neg: 3,785,871 | 98.04%
N alleles: 192
Total: 34,126,883
Pos: 669,104 | 1.96%
Neg: 33,457,779 | 98.04%
N alleles: 146


In [ ]:
%%time
# create seq_allele dict for pt and ft for our dataframe and assign proper domain_id, then perform ft splits
seq_allele_pt_ft_dict={"pt": ms_train_data_pt.seq_allele.tolist(), "ft": ms_train_data_ft.seq_allele.tolist()}

# add domain id split
ms_train_data=add_domain_id_based_on_pt_ft_split(ms_train_data, seq_allele_pt_ft_dict)

# compute fraction
print(ms_train_data.domain_id.value_counts())
print("Fraction in PT set: ", ms_train_data.domain_id.value_counts()[0]/(ms_train_data.domain_id.value_counts()[0]+ms_train_data.domain_id.value_counts()[1]))

domain_id
1    34126883
0     3861640
Name: count, dtype: int64
Fraction in PT set:  0.10165280708597173
CPU times: user 25.6 s, sys: 1.74 s, total: 27.3 s
Wall time: 27.3 s


In [ ]:
%%time
# crete ft splits
ms_train_data.reset_index(inplace=True, drop=True)
ms_train_data=create_ft_splits(ms_train_data, seq_allele_pt_ft_dict, seed=42)

Total samples: 37,988,523
FT samples: 34,126,883 (89.8%)
Creating 9 FT splits (~11% each)
Processing 146 alleles...


Splitting alleles: 100%|██████████| 146/146 [03:43<00:00,  1.53s/it]



Split distribution:
  Split 0: 3,861,640 samples (10.2%)
  Split 1: 3,794,933 samples (10.0%)
  Split 2: 3,791,813 samples (10.0%)
  Split 3: 3,791,140 samples (10.0%)
  Split 4: 3,792,505 samples (10.0%)
  Split 5: 3,792,346 samples (10.0%)
  Split 6: 3,791,071 samples (10.0%)
  Split 7: 3,792,550 samples (10.0%)
  Split 8: 3,791,785 samples (10.0%)
  Split 9: 3,788,740 samples (10.0%)
CPU times: user 4min 11s, sys: 5.05 s, total: 4min 16s
Wall time: 4min 16s


In [ ]:
# get the seq_allele dict for the splits (note this contains unpadded data)
ms_train_data_seq_allele_dict_10_pct_splits=build_split_dict(ms_train_data, split_col='ft_split', seq_allele_col='seq_allele')

In [ ]:
"""
Save the splits
"""
# save_dict_pickle(ms_train_data_seq_allele_dict_10_pct_splits, f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/ms_train_data_seq_allele_dict_10_pct_splits.pkl")

### MS cluster-specific train and test sets

In [ ]:
%%time
# read train and test sets for clustering specifically
ms_train_data_for_clustering=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/ms_train_data_for_clustering.csv", index_col=0)

CPU times: user 12.4 s, sys: 2.42 s, total: 14.9 s
Wall time: 14.9 s


In [ ]:
# process
ms_train_data_for_clustering=full_process_data(ms_train_data_for_clustering, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_train_data_for_clustering=create_seq_allele_column(ms_train_data_for_clustering)

In [ ]:
# 9-mers only
ms_train_data_for_clustering["peptide"]=ms_train_data_for_clustering.seq.str.replace("$", "") # replace any padding if added
ms_train_data_for_clustering["seq_len"]=ms_train_data_for_clustering.peptide.apply(len)
ms_train_data_for_clustering_9mers=ms_train_data_for_clustering.query("seq_len == 9")

In [ ]:
%%time
# for A0201
df_louv, js_mat_louv = louvain_cluster_peptides(
    df=ms_train_data_for_clustering_9mers.query("label == 1 and allele == 'A0201'"),
    peptide_col="peptide",
    length=9,
    feature_table=processed_feature_table, 
    out_cluster_col="louvain_id",
    knn_k=30,
    knn_metric="cosine",
    symmetrize="mutual",
    resolution=1.2,
    small_cluster_min_count=50,
    small_cluster_id=-1,
    compute_js_matrix=True,
    js_min_cluster_count=100,
    js_alpha=0.5,
    js_agg="mean",
    device="cuda",
)

[louvain] n=32341 peptides | L=9 | metric=cosine | k=30 | device=cuda
[louvain] Encoding peptides -> embedding matrix ...
[louvain] Computing kNN ...


kNN (cosine) chunks: 100%|██████████| 16/16 [00:00<00:00, 96.02it/s]

[louvain] Building sparse graph (symmetrize=mutual) ...


[louvain] Graph edges: 546,594
[louvain] Running Louvain (resolution=1.2) ...
[louvain] Using igraph+leidenalg backend
[louvain] Grouping 44 clusters with n<50 into id=-1
[js] Computing PWM-JS matrix (min_cluster_count=100, alpha=0.5, agg=mean)


[js] pairwise: 100%|██████████| 17/17 [00:00<00:00, 464.72it/s]

CPU times: user 10.3 s, sys: 191 ms, total: 10.5 s
Wall time: 10.4 s


In [ ]:
# get a0201 negatives
allele="A0201"
print(allele)
ms_train_data_for_clustering_9mers_allele_negs=ms_train_data_for_clustering_9mers.query(f"allele == '{allele}' and label == 0")
print(ms_train_data_for_clustering_9mers_allele_negs.shape)

A0201
(1617558, 5)


In [ ]:
# get a0201 specific train and test data
allele_train_pos_df, allele_train_neg_df, joint_ext_test_df=sample_pos_neg_per_cluster(df_louv, ms_train_data_for_clustering_9mers_allele_negs, 150, min_count=100, ratio=99, peptide_col='peptide', cluster_col='louvain_id', pos_label=1, neg_label=0, seed=42)

In [ ]:
# create dict for louvain id to seq allele, then given this, and the main df, we remove any seq allele within our list of louvain ids from main train (empty list means we remove nothing). 
allele_train_pos_df=create_seq_allele_column(allele_train_pos_df)

allele_train_pos_louvain_id_to_seq_allele=cluster_to_seq_alleles(allele_train_pos_df, cluster_col='louvain_id', seq_allele_col='seq_allele')
# print(allele_train_pos_louvain_id_to_seq_allele.keys())

In [ ]:
%%time
"""
Save for A0201
"""
os.makedirs(g"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/A0201", exist_ok=True)

# joint_ext_test_df.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/A0201/cluster_specific_test_set.csv")

# allele_train_pos_df.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/A0201/allele_train_pos_df.csv")

# save_dict_pickle(allele_train_pos_louvain_id_to_seq_allele, f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/A0201/allele_train_pos_louvain_id_to_seq_allele.pkl")

In [ ]:
%%time
# for other alleles
alleles_to_cluster=["A0301", "B2705", "B4002", "C0501", "C1202"] # "A0201" done above

ms_train_data_for_clustering_9mers["peptide"]=ms_train_data_for_clustering_9mers.seq.str.replace("$", "")

for target_allele in alleles_to_cluster:
    print(target_allele)
    
    df_allele_to_cluster_pos_9mers=ms_train_data_for_clustering_9mers.query(f"allele == '{target_allele}' and label == 1")
    print(df_allele_to_cluster_pos_9mers.shape)

    # get louvain clustering
    df_louv_allele, js_mat_louv_allele = louvain_cluster_peptides(
        df=df_allele_to_cluster_pos_9mers,
        peptide_col="peptide",
        length=9,
        feature_table=processed_feature_table, 
        out_cluster_col="louvain_id",
        knn_k=30,
        knn_metric="cosine",
        symmetrize="mutual",
        resolution=1.2,
        small_cluster_min_count=50,
        small_cluster_id=-1,
        compute_js_matrix=True,
        js_min_cluster_count=100,
        js_alpha=0.5,
        js_agg="mean",
        device="cuda",
    )

    ms_train_data_for_clustering_9mers_allele_negs=ms_train_data_for_clustering_9mers.query(f"allele == '{target_allele}' and label == 0")
    print(ms_train_data_for_clustering_9mers_allele_negs.shape)

    new_pos_df_allele, new_neg_df_allele, joint_ext_test_df_allele=sample_pos_neg_per_cluster_percent(df_louv_allele, ms_train_data_for_clustering_9mers_allele_negs, percent=0.1, min_samples=30, max_samples=150, min_count=100, ratio=99, peptide_col='peptide', cluster_col='louvain_id', pos_label=1, neg_label=0, seed=42)

    # create dict for louvain id to seq allele
    new_pos_df_allele=create_seq_allele_column(new_pos_df_allele)

    allele_train_pos_louvain_id_to_seq_allele=cluster_to_seq_alleles(new_pos_df_allele, cluster_col='louvain_id', seq_allele_col='seq_allele')
    # print(allele_train_pos_louvain_id_to_seq_allele.keys())

    # save dataframes
    
    # os.makedirs(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}", exist_ok=True)

    # joint_ext_test_df.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/cluster_specific_test_set.csv")

    # new_pos_df_allele.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/allele_train_pos_df.csv")

    # save_dict_pickle(allele_train_pos_louvain_id_to_seq_allele, f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/allele_train_pos_louvain_id_to_seq_allele.pkl")

In [ ]:
%%time
# get background
chp_filtered_negative_peptides=generate_kmers_from_proteome(canonical_human_proteins, kmer_lengths=[8,9,10,11,12,13,14])
random_negative_peptides=generate_random_peptides(n_peptides=10, lengths=[8,9,10,11,12,13,14]) 

chp_filtered_negative_peptides_no_dup={}
for l,p in tqdm(chp_filtered_negative_peptides.items()):
    chp_filtered_negative_peptides_no_dup[l]=list(set(p))
    print("Length", l)
    print(len(chp_filtered_negative_peptides[l]))
    print(len(chp_filtered_negative_peptides_no_dup[l]))

Generating kmers from proteome:   0%|          | 0/20360 [00:00<?, ?it/s]

 29%|██▊       | 2/7 [00:04<00:11,  2.24s/it]

Length 8
11218564
10275287
Length 9
11198183
10378333


 43%|████▎     | 3/7 [00:06<00:08,  2.23s/it]

Length 10
11177802
10438243


...

Length 12
11137041
10510945


 86%|████████▌ | 6/7 [00:13<00:02,  2.25s/it]

Length 13
11116662
10533432


100%|██████████| 7/7 [00:15<00:00,  2.27s/it]

Length 14
11096285
10549606
CPU times: user 1min 1s, sys: 3.77 s, total: 1min 5s
Wall time: 1min 5s


In [ ]:
%%time
# get cluster-specific negatives test df

allele_list=["A0201", "A0301", "B2705", "B4002", "C0501", "C1202"]

for target_allele in allele_list:
    print(target_allele)

    allele_train_pos_df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/allele_train_pos_df.csv", index_col=0)

    joint_ext_test_df=pd.read_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/cluster_specific_test_set.csv", index_col=0) 

    # remove sequences that overlap
    # for the target allele
    ms_train_data_for_clustering["seq_len"]=ms_train_data_for_clustering.seq.str.replace("$", "").apply(len)

    target_allele_all_9mer_sequences = ms_train_data_for_clustering.query(f"allele == '{target_allele}' and seq_len == 9").seq.str.replace("$", "").tolist()

    chp_filtered_negative_peptides_no_dup_9mers_cleaned=remove_sequences(chp_filtered_negative_peptides_no_dup[9], target_allele_all_9mer_sequences)

    # get log odds
    allele_train_pos_df["peptide"]=allele_train_pos_df.seq.str.replace("$", "")
    synthetic_negs_w_louvain_id=assign_peptides_to_clusters_logodds_gpu(
        allele_train_pos_df,
        chp_filtered_negative_peptides_no_dup_9mers_cleaned,
        peptide_col = "peptide",
        cluster_col = "louvain_id",
        background_peptides_by_length = {9: chp_filtered_negative_peptides_no_dup_9mers_cleaned},
        aa_alphabet = AA20_ASSIGN,
        pseudocount_bg = 1.0,
        pseudocount_ref = 50.0,
        device = "cuda",
        prob_prefix = "cluster_prob_",
        return_logodds = False,
    )

    # assign target allele and full process
    synthetic_negs_w_louvain_id["seq"]=synthetic_negs_w_louvain_id["peptide"].tolist()
    synthetic_negs_w_louvain_id["allele"]=target_allele
    synthetic_negs_w_louvain_id_processed=full_process_data(synthetic_negs_w_louvain_id, mhc_seq_dict, remove_flank=False, add_pseudo_label=False)

    joint_ext_test_df_resampled=resample_negatives_from_df2(joint_ext_test_df, synthetic_negs_w_louvain_id_processed, "louvain_id", seq_col='seq', label_col='label', seed=42)

    # save df

    # joint_ext_test_df_resampled.to_csv(f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/Cluster_Specific_Analysis/{target_allele}/cluster_specific_test_set_hard_negatives.csv")

### MS allele holdout sets

In [ ]:
%%time
# read in ms train set
ms_train_test_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{ms_train_test_dir}/ms_train_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{ms_train_test_dir}/ms_test_data.csv", index_col=0)
ms_train_data.reset_index(inplace=True, drop=True)
ms_test_data.reset_index(inplace=True, drop=True)

CPU times: user 36.8 s, sys: 7.05 s, total: 43.9 s
Wall time: 43.9 s


In [ ]:
# filter to test alleles only
ms_train_data_test_alleles=ms_train_data[ms_train_data.allele.isin(ms_test_data.allele.unique().tolist())]

In [ ]:
# get summary of counts
ms_train_data_test_alleles_summary=summarize_pos_neg_per_allele(ms_train_data_test_alleles, allele_col='allele', label_col='label')

In [ ]:
# filtered counts
ms_train_data_test_alleles_summary_filtered=ms_train_data_test_alleles_summary.query("n_pos <= 10000 and n_pos >= 1000")
allele_set_to_select_from=ms_train_data_test_alleles_summary_filtered.allele.tolist()

In [ ]:
# get balanced allele splits
# allele_holdout_splits=make_random_allele_sets(allele_set_to_select_from, n_sets=10, n_per_set=10, seed=42)
allele_holdout_splits=make_balanced_allele_sets(allele_set_to_select_from, n_sets=10, n_per_set=10, seed=42)

In [ ]:
print(allele_holdout_splits)

{1: ['C0802', 'B0704', 'A0206', 'A3101', 'A0302', 'B3802', 'B4405', 'C0801', 'B5501', 'C0102'], 2: ['B4701', 'A2301', 'C0303', 'C1502', 'B5201', 'B1402', 'A7401', 'B2707', 'A6601', 'B2701'], 3: ['C0202', 'B1503', 'B4901', 'A2601', 'A1102', 'C1701', 'B3501', 'B2709', 'B8101', 'B1517'], 4: ['B5401', 'B3503', 'C0602', 'B2708', 'B5502', 'A3001', 'B4006', 'B2706', 'C0701', 'A0252'], 5: ['C0302', 'B4032', 'A3002', 'A3402', 'B4201', 'A3303', 'B1302', 'A0204', 'A2407', 'B1801'], 6: ['B5001', 'B3507', 'B5301', 'B3801', 'A6802', 'B3701', 'A3401', 'C0403', 'B5801', 'B1301'], 7: ['B4501', 'C0702', 'A0202', 'B1511', 'B1510', 'A0205', 'B5703', 'B3508', 'A0211', 'B4101'], 8: ['C0401', 'B3502', 'B4601', 'A3301', 'A2902', 'C0704', 'B3901', 'B5601', 'B2704', 'A0207'], 9: ['B2702', 'C1601', 'C1402', 'A3601', 'B1518', 'A2501', 'C1403', 'B3905', 'A3201', 'C1203'], 10: ['C1402', 'B1801', 'A3601', 'B3906', 'A0203', 'A1102', 'B1510', 'C1403', 'B4006', 'A3201']}


In [ ]:
# save the balanced allele splits
# save_dict_pickle(allele_holdout_splits, f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets/allele_holdout_splits.pkl")

### Human and Swiss-Prot proteome random background for ranking and feature interpretability

#### Human proteome background for ranking

In [ ]:
%%time
# generate synthetic negatives from human proteome
chp_filtered_negative_peptides=generate_kmers_from_proteome(canonical_human_proteins, kmer_lengths=[8,9,10,11,12,13,14])
random_negative_peptides=generate_random_peptides(n_peptides=10, lengths=[8,9,10,11,12,13,14])

# remove all duplicate sequences for now
chp_filtered_negative_peptides_no_dup={}
for l,p in tqdm(chp_filtered_negative_peptides.items()):
    chp_filtered_negative_peptides_no_dup[l]=list(set(p))
    print("Length", l)
    print(len(chp_filtered_negative_peptides[l]))
    print(len(chp_filtered_negative_peptides_no_dup[l]))

 29%|██▊       | 2/7 [00:06<00:14,  2.99s/it]

Length 8
11250173
10300888
Length 9
11229814
10405224


 43%|████▎     | 3/7 [00:08<00:10,  2.74s/it]

Length 10
11209455
10466109


 57%|█████▋    | 4/7 [00:11<00:07,  2.63s/it]

...

Length 12
11168738
10540392


 86%|████████▌ | 6/7 [00:15<00:02,  2.54s/it]

Length 13
11148381
10563518


100%|██████████| 7/7 [00:18<00:00,  2.64s/it]

Length 14
11128026
10580237
CPU times: user 1min 1s, sys: 5.04 s, total: 1min 6s
Wall time: 1min 6s


In [ ]:
# get ms train set for sequence removal
datasets_dir=f"{DATA_DIR}/Training_Datasets/Processed/MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0)

In [ ]:
# filter ms train sequences
exclude_seqs=ms_train_data.seq.unique().tolist()
chp_filtered_negative_peptides_no_dup_cleaned=filter_dict_by_peptides(chp_filtered_negative_peptides_no_dup, exclude_seqs)

In [ ]:
%%time
# background generation

# run_background_gen=False
run_background_gen=True

if run_background_gen:

    # use larger allele set
    alleles_for_background=['A6802', 'A6801', 'B5301', 'B5501', 'C0202', 'A2501', 'B1302', 'B5105', 'B4601', 'A0201', 'B1502', 'C0401', 'A0101', 'A2601', 'B5702', 'B0801', 'C1701', 'A3303', 'A0301', 'B4102', 'B4402', 'B4901', 'B2702', 'C0722', 'B5101', 'C0602', 'B3502', 'C0102', 'B0702', 'A3102', 'B1805', 'B3503', 'C1601', 'C1203', 'A2902', 'A3001', 'C0303', 'A0203', 'C1801', 'C0801', 'B1402', 'B4405', 'B4101', 'C0501', 'C0304', 'C1402', 'B4001', 'B2705', 'C0802', 'B4403', 'C1604', 'B1803', 'B5701', 'C0701', 'A3101', 'B3801', 'A2402', 'C0702', 'B1801', 'B1401', 'A2301', 'B3508', 'C1502', 'A3201', 'B1501', 'A1101', 'B5601', 'B5108', 'B2708', 'A2603', 'B2701', 'B4105', 'A1102', 'C1215', 'A0103', 'A0217', 'A3402', 'A0252', 'C0313', 'A6901', 'B5201', 'B2706', 'B3901', 'B5001', 'A0125', 'C0302', 'A2608', 'B4408', 'B1524', 'B3507', 'B3504', 'B4106', 'B1515', 'A6602', 'B4002', 'B1508', 'B5401', 'A3014', 'A2901', 'A2602', 'B6701', 'B1403', 'C0711', 'A3301', 'B2713', 'A0207', 'B4032', 'B4103', 'B2709', 'B1542', 'B1511', 'B2707', 'B4801', 'A3202', 'C1202', 'B4501', 'B1513', 'B3701', 'B1504', 'B3557', 'A0205', 'B3909', 'C1505', 'B1517', 'A0307', 'B3814', 'B2703', 'A0202', 'A3002', 'B3905', 'B4428', 'B3802', 'B3924', 'C0814', 'A0214', 'B0704', 'A7401', 'B4104', 'B4409', 'A2407', 'B1516', 'A2503', 'B5002', 'B1503', 'B5703', 'B4201', 'A0206', 'C0706', 'B5801', 'C1403', 'B2710', 'A2413', 'C0403', 'B0815', 'B0705', 'B1509', 'B1301', 'A6601', 'B5502', 'C1602', 'A2403', 'B3501', 'B3906', 'B1518', 'B8101', 'B5802', 'C1513', 'B7301', 'B2716', 'B4427', 'B4701', 'A0204', 'B5104', 'A0302', 'A3004', 'B1510', 'C0301', 'C0704', 'A0211', 'A3401', 'A0220', 'A2406', 'A2913', 'B4006', 'B4440', 'B5102', 'B3910', 'A3003', 'B3506', 'A2914', 'B0706', 'A3022', 'B2704', 'C1204', 'C0106', 'A3601', 'A1110']

    alleles_for_background=list(set(alleles_for_background))
    print("N alleles: ", len(alleles_for_background))
    lengths_for_background=[8,9,10,11,12,13,14]

    # 10k for each
    n_sample=10_000

    chp_background_10k_per_allele_x_len=sample_background_distribution(chp_filtered_negative_peptides_no_dup_cleaned, alleles_for_background, lengths_for_background, n_sample, seed=42)

print(chp_background_10k_per_allele_x_len.shape)

N alleles:  197


Sampling background: 100%|██████████| 1379/1379 [00:29<00:00, 47.18it/s]


(13790000, 3)
CPU times: user 28.4 s, sys: 2.04 s, total: 30.4 s
Wall time: 30.4 s


In [ ]:
%%time
# full process and assign pseudolabel
if run_background_gen:
    # ---
    chp_background_10k_per_allele_x_len_processed=full_process_data(chp_background_10k_per_allele_x_len, mhc_seq_dict, remove_flank=False, add_pseudo_label=True)

    chp_background_10k_per_allele_x_len_processed["peptide"]=chp_background_10k_per_allele_x_len_processed.seq.str.replace("$", "")

    chp_background_10k_per_allele_x_len_processed=create_seq_allele_column(chp_background_10k_per_allele_x_len_processed)

CPU times: user 13.4 s, sys: 1.88 s, total: 15.2 s
Wall time: 15.2 s


In [ ]:
# drop duplicate seq allele
chp_background_10k_per_allele_x_len_processed.drop_duplicates(subset=["seq_allele"], inplace=True)
print(chp_background_10k_per_allele_x_len_processed.shape)

(13774791, 15)


'\n(13783547, 15)\n\n'

In [ ]:
%%time
# build cache for the background df

# build_caches=True
build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.pt"
    build_token_cache(chp_background_10k_per_allele_x_len_processed, data_to_run_path, tokenizer, use_domain_id=False)

CPU times: user 15 µs, sys: 2 µs, total: 17 µs
Wall time: 32.7 µs


In [ ]:
%%time
"""
Save background peptides.
"""
# chp_background_10k_per_allele_x_len_processed.to_csv(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.csv")

CPU times: user 33.2 s, sys: 2.29 s, total: 35.5 s
Wall time: 35.5 s


#### Human proteome background for EpiScan feature analysis

In [ ]:
%%time
# generate synthetic negatives from human proteome
chp_filtered_negative_peptides=generate_kmers_from_proteome(canonical_human_proteins, kmer_lengths=[8,9,10,11,12,13,14])
random_negative_peptides=generate_random_peptides(n_peptides=10, lengths=[8,9,10,11,12,13,14]) 

chp_filtered_negative_peptides_no_dup={}
for l,p in tqdm(chp_filtered_negative_peptides.items()):
    chp_filtered_negative_peptides_no_dup[l]=list(set(p))
    print("Length", l)
    print(len(chp_filtered_negative_peptides[l]))
    print(len(chp_filtered_negative_peptides_no_dup[l]))

 29%|██▊       | 2/7 [00:05<00:12,  2.50s/it]

Length 8
11250173
10300888
Length 9
11229814
10405224


 43%|████▎     | 3/7 [00:07<00:09,  2.50s/it]

Length 10
11209455
10466109


 57%|█████▋    | 4/7 [00:10<00:07,  2.51s/it]

...

Length 12
11168738
10540392


 86%|████████▌ | 6/7 [00:15<00:02,  2.53s/it]

Length 13
11148381
10563518


100%|██████████| 7/7 [00:17<00:00,  2.52s/it]

Length 14
11128026
10580237
CPU times: user 1min, sys: 4.87 s, total: 1min 4s
Wall time: 1min 4s


In [ ]:
# background gen
# run_background_gen=False
run_background_gen=True

if run_background_gen:
    alleles_for_background=["A0201", "A0301", "B0801", "B5701"] # for episcan
    lengths_for_background=[9]
    n_sample=100_000

    chp_background_for_episcsan=sample_background_distribution(chp_filtered_negative_peptides_no_dup, alleles_for_background, lengths_for_background, n_sample)

Sampling background: 100%|██████████| 4/4 [00:05<00:00,  1.39s/it]


In [ ]:
if run_background_gen:
    chp_background_for_episcsan_processed=full_process_data(chp_background_for_episcsan, mhc_seq_dict, remove_flank=False, add_pseudo_label=True)

    chp_background_for_episcsan_processed["peptide"]=chp_background_for_episcsan_processed.seq.str.replace("$", "")

    chp_background_for_episcsan_processed=create_seq_allele_column(chp_background_for_episcsan_processed)

    chp_background_for_episcsan_processed.drop_duplicates(subset=["seq_allele"], inplace=True)

In [ ]:
%%time
"""
Save background peptides
"""
# chp_background_for_episcsan_processed.to_csv(f"{DATA_DIR}/chp_background_for_episcsan_processed.csv")

CPU times: user 3.11 s, sys: 372 ms, total: 3.49 s
Wall time: 3.48 s


#### SP background for YD Y84A analysis

In [ ]:
# get proteome
swissprot_proteome=pd.read_csv(f"{DATA_DIR}/swissprot_proteome.csv", index_col=0)

In [ ]:
# get holec alleles
holec_alleles=['A0201', 'A1101', 'A0301', 'A0101', 'C0401', 'A2402', 'C0102', 'C1202', 'C0303', 'B3502', 'B3503', 'B3901', 'A3002', 'A3001', 'B5801', 'B5502']

In [ ]:
%%time
# create the background for y84a analysis
num_sample=100_000
lengths_to_sample=[9]

swissprot_proteome_background_for_holec=sample_background_peptides(swissprot_proteome, num_sample, lengths_to_sample)

swissprot_proteome_background_for_holec_df = pd.concat([
    pd.DataFrame({'seq': peptides, 'seq_len': length})
    for length, peptides in swissprot_proteome_background_for_holec.items()
], ignore_index=True)

swissprot_proteome_background_for_holec_df=assign_alleles(swissprot_proteome_background_for_holec_df, holec_alleles, allele_col='allele')

swissprot_proteome_background_for_holec_df_processed=full_process_data(swissprot_proteome_background_for_holec_df, mhc_seq_dict, remove_flank=False, add_pseudo_label=True)

swissprot_proteome_background_for_holec_df_processed["peptide"]=swissprot_proteome_background_for_holec_df_processed.seq.str.replace("$", "")

swissprot_proteome_background_for_holec_df_processed=create_seq_allele_column(swissprot_proteome_background_for_holec_df_processed)

swissprot_proteome_background_for_holec_df_processed.drop_duplicates(subset=["seq_allele"], inplace=True)

swissprot_proteome_background_for_holec_df_processed.reset_index(inplace=True, drop=True)

print(swissprot_proteome_background_for_holec_df_processed.allele.value_counts())
print(swissprot_proteome_background_for_holec_df_processed.shape)

Cleaning sequences (removing non-standard amino acids)...


Processing Lengths: 100%|██████████| 1/1 [00:00<00:00,  5.76it/s]


allele
A0201    99771
A1101    99771
A0301    99771
A0101    99771
C0401    99771
A2402    99771
C0102    99771
C1202    99771
C0303    99771
B3502    99771
B3503    99771
B3901    99771
A3002    99771
A3001    99771
B5801    99771
B5502    99771
Name: count, dtype: int64
(1596336, 15)
CPU times: user 8.46 s, sys: 232 ms, total: 8.69 s
Wall time: 8.69 s


In [ ]:
%%time
# build cache for the sp df

# build_caches=True
build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/swissprot_proteome_background_for_holec_df_processed.pt"

    build_token_cache(swissprot_proteome_background_for_holec_df_processed, data_to_run_path, tokenizer, use_domain_id=False)

CPU times: user 13 µs, sys: 2 µs, total: 15 µs
Wall time: 26.2 µs


In [ ]:
%%time
"""
Save background peptides
"""
# swissprot_proteome_background_for_holec_df_processed.to_csv(f"{DATA_DIR}/swissprot_proteome_background_for_holec_df_processed.csv")

CPU times: user 3.78 s, sys: 292 ms, total: 4.07 s
Wall time: 4.07 s
